> **This notebook is generated.** The canonical version of this project lives in its GitHub repo — [agentic-rag-research-assistant](https://github.com/Cena-S/agentic-rag-research-assistant) — and this file is produced from the repo's capstone notebook so the two can never disagree. If something here looks wrong, the repo is the thing to fix.


In [ ]:
# === GENERATED: inlined from src/ - do not edit, regenerate ===
#
# Kaggle cannot import this project's `src/` package, so every module the
# notebook uses is inlined below and registered in sys.modules. The cells that
# follow are byte-identical to the GitHub capstone.
#
# Canonical source: the repo. Regenerate with 05-communication/tools/make_kaggle_notebook.py

import sys, types

# Kaggle's writable dir stands in for the repo root, so modules that resolve paths
# from __file__ (PROJECT_ROOT = Path(__file__).parent.parent) still work.
_ROOT = '/kaggle/working'

_src = types.ModuleType('src'); _src.__path__ = [_ROOT + '/src']
_src.__file__ = _ROOT + '/src/__init__.py'
sys.modules['src'] = _src

# ---- src/config.py --------------------------------------------------
_m = types.ModuleType('src.config')
_m.__dict__['__name__'] = 'src.config'
_m.__dict__['__file__'] = _ROOT + '/src/config.py'
_m.__dict__['__package__'] = 'src'
_code_config = r'''"""Central configuration for the agentic RAG research assistant.

Every judgement parameter in this project is a named constant here, each with a
prose "why". The rule (borrowed from the rest of this portfolio): if a number
changes the answer, it must be visible and defensible — never a magic number
buried in a function. `run_pipeline.py`, `run_eval.py`, the notebooks, and the
tests all read their settings from this one module, so there is a single source
of truth for how the system is tuned.
"""

from __future__ import annotations

import os
from pathlib import Path

# --------------------------------------------------------------------------- #
# Determinism
# --------------------------------------------------------------------------- #
# One seed for the whole project. Embedding a query, sampling papers, shuffling
# an evaluation set — all of it is reproducible so every published number can be
# regenerated exactly. Matches the SEED = 42 convention used across the portfolio.
SEED = 42

# --------------------------------------------------------------------------- #
# Paths — everything is resolved relative to the project root, so the code runs
# the same whether invoked from a notebook, a test, or the command line.
# --------------------------------------------------------------------------- #
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
INDEX_DIR = DATA_DIR / "index"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
RESULTS_DIR = OUTPUTS_DIR / "results"
EVAL_DIR = PROJECT_ROOT / "eval"

# Named artifact files. Referenced by the pipeline and every consumer so a
# rename happens in exactly one place.
PROVENANCE_FILE = RAW_DIR / "provenance.json"
PAPERS_FILE = RAW_DIR / "papers.jsonl"          # one arXiv paper per line
CHUNKS_FILE = PROCESSED_DIR / "chunks.parquet"  # one chunk per row
FAISS_INDEX_FILE = INDEX_DIR / "faiss.index"    # the serialized vector index
CHUNK_META_FILE = INDEX_DIR / "chunk_meta.parquet"  # row-aligned with the index

# The retrieval benchmark's own artifacts. The generated question set is
# committed (it is small, and every published number depends on it); the drift
# slice is raw downloaded data and is git-ignored like the rest of data/.
KNOWN_ITEM_FILE = EVAL_DIR / "known_item_set.json"
DRIFT_PAPERS_FILE = RAW_DIR / "papers_drift.jsonl"
DRIFT_PROVENANCE_FILE = RAW_DIR / "provenance_drift.json"
ARM_ROWS_FILE = RESULTS_DIR / "arm_comparison.json"   # one row per (arm, query)
ARM_SUMMARY_FILE = RESULTS_DIR / "arm_summary.json"   # per-arm means + intervals
SIGNIFICANCE_FILE = RESULTS_DIR / "arm_significance.json"  # pairwise paired tests
K_SWEEP_FILE = RESULTS_DIR / "k_sweep.json"           # nDCG/recall vs cutoff
CHUNK_ABLATION_FILE = RESULTS_DIR / "chunk_ablation.json"  # quality vs chunk size
ABLATION_SIGNIFICANCE_FILE = RESULTS_DIR / "chunk_ablation_significance.json"
DRIFT_RESULTS_FILE = RESULTS_DIR / "drift.json"

# --------------------------------------------------------------------------- #
# Corpus acquisition (arXiv)
# --------------------------------------------------------------------------- #
# We retrieve recent NLP / ML papers. cs.CL is computational linguistics and
# cs.LG is machine learning — the two categories most likely to contain the
# retrieval, embedding, and agent literature our evaluation questions ask about,
# so the corpus actually *can* answer them.
ARXIV_CATEGORIES = ("cs.CL", "cs.LG")

# How many papers to pull. Bounded deliberately: the whole pipeline (download →
# chunk → embed → index) must finish in a few minutes on a CPU laptop so the
# capstone notebook can run top-to-bottom. A few thousand abstracts is plenty to
# make retrieval quality visible without a GPU or an overnight job.
MAX_PAPERS = 2000

# arXiv asks clients to page politely and sleep between requests. These match
# their published guidance; raising the page size or dropping the delay risks a
# temporary IP block, which would make the pipeline *less* reproducible.
ARXIV_PAGE_SIZE = 100
ARXIV_REQUEST_DELAY_SECONDS = 3.0

# Seminal papers, fetched by id and merged into the corpus regardless of the
# recent slice. Two reasons this matters:
#   1. The evaluation set's ground-truth answers point at these papers, so they
#      must be present for the metrics to be meaningful. Pinning them by id makes
#      the evaluation deterministic — it does not depend on which recent papers
#      happened to be returned that day.
#   2. They are the papers the retrieval/embedding/agent literature is *about*,
#      so a corpus that contains them can actually answer questions about the
#      field's foundations.
SEED_PAPER_IDS = (
    "1706.03762",  # Attention Is All You Need (the Transformer)
    "2005.11401",  # Retrieval-Augmented Generation (the original RAG paper)
    "1810.04805",  # BERT
    "2005.14165",  # GPT-3 (Language Models are Few-Shot Learners)
    "1908.10084",  # Sentence-BERT (the embedding model family we use)
    "2004.04906",  # Dense Passage Retrieval (DPR)
    "2210.03629",  # ReAct (reason + act — the agentic-loop idea)
    "2201.11903",  # Chain-of-Thought prompting
    "1301.3781",   # word2vec (Efficient Estimation of Word Representations)
    "1409.0473",   # Bahdanau attention (align-and-translate)
    "2203.02155",  # InstructGPT (training LMs to follow instructions with RLHF)
    "2302.13971",  # LLaMA
)

# --------------------------------------------------------------------------- #
# Chunking
# --------------------------------------------------------------------------- #
# Retrieval works on *chunks*, not whole documents. Too-large chunks dilute the
# embedding (one vector trying to represent several ideas); too-small chunks lose
# the context a sentence needs to be meaningful. ~1000 characters is a few
# sentences — one coherent idea — which is a well-trodden default for abstract-
# and-paragraph-length scientific text.
CHUNK_SIZE_CHARS = 1000

# Overlap carries a little context across the boundary so a fact split across two
# chunks is still retrievable from at least one of them. 150 chars (~15%) is the
# usual trade-off: enough to avoid orphaning a sentence, small enough not to
# balloon the index with near-duplicate text.
CHUNK_OVERLAP_CHARS = 150

# --------------------------------------------------------------------------- #
# Embeddings
# --------------------------------------------------------------------------- #
# all-MiniLM-L6-v2 is the workhorse sentence-transformer: 384-dimensional, fast
# on CPU, and strong on short-text semantic similarity — exactly this task. It is
# small enough to download and run in the capstone without a GPU, which keeps the
# whole retrieval path reproducible on any machine.
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM = 384
EMBED_BATCH_SIZE = 64

# --------------------------------------------------------------------------- #
# Retrieval
# --------------------------------------------------------------------------- #
# How many chunks a single retrieval call returns. 5 is a common default: enough
# context for the LLM to synthesize an answer, few enough that an irrelevant
# chunk can't crowd out the relevant ones ("lost in the middle"). The agent can
# retrieve multiple *times*; this is the width of one retrieval.
TOP_K = 5

# --------------------------------------------------------------------------- #
# Retrieval benchmark — the four arms
# --------------------------------------------------------------------------- #
# The cutoff the benchmark reports at. TOP_K = 5 is what the *system* feeds the
# LLM; EVAL_K = 10 is what the *evaluation* measures over. They are deliberately
# different numbers: nDCG@10 is the field's convention (BEIR, MS MARCO, TREC all
# report at 10), and measuring deeper than you serve shows whether a relevant
# document was merely ranked 7th rather than missing altogether — which is the
# difference between "retrieval failed" and "the cutoff is too tight".
EVAL_K = 10

# --- BM25 (the lexical arm) ------------------------------------------------ #
# BM25's term-frequency saturation. Raising k1 lets repeated terms keep adding
# score; lowering it makes one occurrence nearly as good as five. 1.5 is the
# long-standing default from the TREC literature and the value Lucene and
# Elasticsearch ship, so the lexical baseline is the one a reviewer expects
# rather than a tuned-to-flatter-itself variant.
BM25_K1 = 1.5

# BM25's length normalisation, in [0, 1]. b = 1 fully penalises long documents,
# b = 0 ignores length entirely. 0.75 is again the Lucene/TREC default. Our
# chunks are near-uniform in length by construction (CHUNK_SIZE_CHARS), so this
# parameter does comparatively little here — worth knowing before claiming BM25
# was carefully tuned. It was not; it is the honest out-of-the-box baseline.
BM25_B = 0.75

# --- Hybrid (reciprocal-rank fusion) --------------------------------------- #
# RRF combines two ranked lists by summing 1 / (RRF_K + rank) across them. The
# constant damps the influence of the very top ranks so one arm's confident-but-
# wrong first hit cannot dominate the fusion. 60 is the value from Cormack et al.
# (2009), where RRF was introduced, and it is what makes fusion work without
# needing the two arms' scores to be on a comparable scale — which dense cosine
# similarity and BM25 emphatically are not.
RRF_K = 60

# --- Cross-encoder reranker ------------------------------------------------ #
# A *cross*-encoder reads the query and the document together and scores the
# pair, instead of embedding each separately and comparing vectors. That joint
# read is why it ranks better — and why it cannot be precomputed into an index,
# so it must run at query time over a candidate shortlist. This is the accuracy/
# latency trade the benchmark exists to measure.
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# How many candidates the reranker rescores. Every candidate is one forward pass,
# so this is the arm's cost dial: too few and the reranker cannot rescue a
# relevant document the first-stage retriever buried; too many and latency grows
# for candidates that were never plausible. 50 is a common first-stage depth and
# is 5x the reported cutoff, so a document has to be badly misranked to be lost
# before reranking gets a chance at it.
RERANK_CANDIDATES = 50

# --- Uncertainty ----------------------------------------------------------- #
# Resamples for the percentile bootstrap. The bootstrap resamples *questions*,
# not documents, because the question set is the sample we generalise over — the
# claim being defended is "on a query like these", not "on this exact list".
# 2000 is enough for a stable 95% interval; more mostly costs time.
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_CI = 0.95

# --- The known-item question set ------------------------------------------- #
# How many papers to sample for generated queries. Each sampled paper yields
# three queries — one per query style (title, abstract sentence, keyword bag) —
# so 60 papers gives 180 graded queries. That is enough for a bootstrap interval
# to be worth printing, which the original 11 hand-written questions were not.
# Sampling is seeded, so the set is identical on every machine.
KNOWN_ITEM_PAPERS = 60

# Length of a keyword query, in content words. Real search queries are short,
# unordered word bags, not grammatical sentences — and a bi-encoder trained on
# sentences has no reason to handle them equally well. Five words is short enough
# to be a realistic search box entry and long enough to identify one paper out of
# 2,012. This is the harshest query style in the set, deliberately.
KEYWORD_QUERY_WORDS = 5

# The shortest abstract sentence accepted as a query. Very short sentences
# ("We release our code.") identify no particular paper, so they would add noise
# to the ground truth rather than difficulty.
MIN_QUERY_CHARS = 40

# --- Drift probe ----------------------------------------------------------- #
# How many newer papers to pull for the staleness check. These are papers the
# index has never seen; the probe asks how well the existing index covers the
# topics they talk about. Kept small because it is one polite arXiv page-walk.
DRIFT_PAPERS = 200

# --- The re-indexing trigger, and the mistake it took a re-index to find ----- #
#
# Drift is measured as the drop in mean top-1 similarity between queries drawn
# from *indexed* papers and queries drawn from *newer, unseen* ones. The original
# trigger fired on a raw 15% relative drop, and on 2026-07-27 the probe read
# 27.2% — so the project recommended re-indexing.
#
# The index was rebuilt on 2026-08-12, from a corpus with **zero** paper overlap
# with the old one (see outputs/results/corpus_turnover.json). The same probe on
# that one-hour-old index read **26.7%**. Rebuilding — the largest freshness
# intervention available — moved the number by half a percentage point.
#
# That is the metric failing, not the index. The drop has a large structural
# floor that has nothing to do with staleness: a baseline query is a sentence
# lifted verbatim out of a document sitting in the index, so its top-1 match is
# near-perfect by construction, while a probe query comes from a document that is
# not in the index and never can be. The gap between "verbatim self-match" and
# "closest topical neighbour" is a property of the measurement, and it is there on
# day zero. A threshold on the raw drop therefore fires on *every* index, always,
# and says nothing.
#
# The floor below is that day-zero reading — measured, not assumed, on the
# freshly-built 2026-08-12 index. It is specific to this corpus, this embedding
# model and this window size, and must be re-measured if any of them change; that
# is the cost of the metric being interpretable at all.
DRIFT_FLOOR_RELATIVE_DROP = 0.267

# What the trigger actually tests: how far the current drop exceeds the day-zero
# floor. This is the "gap widening over successive probes" that drift.py's own
# docstring always described as the signal — the original constant just never
# implemented it. 10 excess points is a judgement call about cost, not a property
# of the data: re-embedding this corpus is a few minutes on a laptop, so it can be
# tight; on an overnight GPU job it would be looser.
DRIFT_TRIGGER_EXCESS_DROP = 0.10

# --------------------------------------------------------------------------- #
# Generation + agent (Claude via the Anthropic API)
# --------------------------------------------------------------------------- #
# The generation model. Sonnet 5 is the cost/quality sweet spot for grounded
# question-answering over retrieved context — near-Opus quality at Sonnet price.
# Swap to "claude-opus-5" for maximum quality or "claude-haiku-4-5" to cut cost;
# every call reads this constant, so it changes in one place.
GEN_MODEL = "claude-sonnet-5"

# The judge model for the answer-quality evaluation (LLM-as-judge). Kept the same
# as the generation model by default so the grader is at least as capable as the
# thing it grades.
JUDGE_MODEL = "claude-sonnet-5"

# Hard ceiling on answer length. Generous enough for a cited multi-paragraph
# answer, bounded so a runaway generation can't rack up cost.
MAX_ANSWER_TOKENS = 1024

# The agentic loop's stopping condition. The agent may call the retrieval tool,
# read what it got, decide it needs more, and search again — but not forever.
# After this many tool-use rounds it must answer with what it has. 4 is enough
# for a two- or three-hop question (decompose → search → search again → answer)
# without letting a confused agent loop indefinitely and burn tokens.
MAX_AGENT_STEPS = 4

# Where the on-disk LLM response cache lives. Every Claude call is cached by a
# hash of its inputs so (a) the notebook and tests reproduce published numbers
# WITHOUT an API key, and (b) re-running the pipeline doesn't pay for identical
# calls twice. This is the same "reference-only, load the saved result"
# convention the portfolio's Kaggle-GPU notebooks use.
LLM_CACHE_FILE = RESULTS_DIR / "llm_cache.json"

# The environment variable holding the Anthropic API key. NEVER hard-code a key;
# NEVER commit one. If it is unset, live LLM calls are skipped and the cache is
# used instead — so the project degrades gracefully to "reproduce from cache".
ANTHROPIC_API_KEY_ENV = "ANTHROPIC_API_KEY"


def api_key_available() -> bool:
    """True when an Anthropic API key is present in the environment.

    Used everywhere a live call is optional: if this is False we fall back to the
    committed response cache instead of erroring, which is what lets the whole
    project reproduce on a machine with no key.
    """
    return bool(os.environ.get(ANTHROPIC_API_KEY_ENV))


def ensure_dirs() -> None:
    """Create every output/data directory this project writes to.

    Called at the top of the pipeline and eval entry points so a fresh checkout
    (where data/ and outputs/ are git-ignored and therefore absent) just works.
    """
    for d in (RAW_DIR, PROCESSED_DIR, INDEX_DIR, FIGURES_DIR, RESULTS_DIR):
        d.mkdir(parents=True, exist_ok=True)
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.config'] = _m
setattr(_src, 'config', _m)
exec(compile(_code_config, 'src/config.py', 'exec'), _m.__dict__)

# ---- src/acquisition.py ---------------------------------------------
_m = types.ModuleType('src.acquisition')
_m.__dict__['__name__'] = 'src.acquisition'
_m.__dict__['__file__'] = _ROOT + '/src/acquisition.py'
_m.__dict__['__package__'] = 'src'
_code_acquisition = r'''"""Acquire the arXiv corpus and record provenance.

This is the first stage of the RAG pipeline. It queries the public arXiv API for
recent cs.CL / cs.LG papers, keeps title + abstract + metadata for every paper,
and pulls full text only where the paper's licence permits redistribution.

The provenance discipline matches the rest of the portfolio: we record the exact
query, the date, the paper count, and a SHA-256 of the saved bytes, so anyone can
see precisely what was downloaded and when — and detect if it silently changed.

Nothing here is committed to git; `run_pipeline.py` fetches it and writes
`data/raw/`, which is git-ignored. The *code* ships, the data does not.
"""

from __future__ import annotations

import hashlib
import json
import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from typing import Iterator

from src import config

# arXiv's Atom API. Documented at https://info.arxiv.org/help/api/index.html.
ARXIV_API_URL = "http://export.arxiv.org/api/query"

# Atom / arXiv XML namespaces, needed to walk the feed with ElementTree.
_NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom",
}

# arXiv licences that permit redistribution of the full text. Papers under these
# licences are safe to store and chunk in full; everything else is kept as
# metadata + abstract only. This is a deliberate, defensible licensing choice —
# not a shortcut — and it is why the corpus is "abstracts for all, full text for
# the CC-licensed subset".
REDISTRIBUTABLE_LICENSES = (
    "http://creativecommons.org/licenses/by/4.0/",
    "http://creativecommons.org/licenses/by-sa/4.0/",
    "http://creativecommons.org/publicdomain/zero/1.0/",
)


@dataclass
class Paper:
    """One arXiv paper as we store it.

    `text` is the field retrieval actually runs over: title + abstract always,
    which is self-contained and licence-safe for every paper. (A future extension
    could append CC-licensed full text here; the `license` field is recorded so
    that decision can be made per-paper and defended.)
    """

    arxiv_id: str
    title: str
    abstract: str
    authors: list[str]
    categories: list[str]
    published: str
    license: str | None

    @property
    def text(self) -> str:
        """The retrievable body: title then abstract, one blank line between."""
        return f"{self.title}\n\n{self.abstract}"

    @property
    def redistributable(self) -> bool:
        """True when the paper's licence permits storing its full text."""
        return self.license in REDISTRIBUTABLE_LICENSES


def _build_query_url(start: int, page_size: int) -> str:
    """Construct one paged arXiv API request URL.

    We search within the configured categories, sorted by submission date
    (newest first) so the corpus is a coherent recent slice rather than an
    arbitrary sample — which makes the "recent NLP/ML literature" framing honest.
    """
    cat_clause = "+OR+".join(f"cat:{c}" for c in config.ARXIV_CATEGORIES)
    params = {
        "search_query": cat_clause,
        "start": str(start),
        "max_results": str(page_size),
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    }
    # arXiv wants the OR-joined category clause un-escaped, so we assemble the
    # query string by hand rather than letting urlencode escape the '+'.
    query = "&".join(f"{k}={urllib.parse.quote(v, safe='+:')}" for k, v in params.items())
    return f"{ARXIV_API_URL}?{query}"


def _parse_feed(xml_bytes: bytes) -> Iterator[Paper]:
    """Yield Paper objects from one Atom feed response."""
    root = ET.fromstring(xml_bytes)
    for entry in root.findall("atom:entry", _NS):
        # The arXiv id is the tail of the entry's <id> URL, minus any version.
        raw_id = entry.findtext("atom:id", default="", namespaces=_NS)
        arxiv_id = raw_id.rsplit("/", 1)[-1].split("v")[0] if raw_id else ""

        title = (entry.findtext("atom:title", default="", namespaces=_NS) or "").strip()
        abstract = (entry.findtext("atom:summary", default="", namespaces=_NS) or "").strip()
        published = entry.findtext("atom:published", default="", namespaces=_NS) or ""

        authors = [
            (a.findtext("atom:name", default="", namespaces=_NS) or "").strip()
            for a in entry.findall("atom:author", _NS)
        ]
        categories = [
            c.attrib.get("term", "")
            for c in entry.findall("atom:category", _NS)
        ]

        # arxiv:license is present only when the author selected a licence.
        license_el = entry.find("arxiv:license", _NS)
        license_url = license_el.text.strip() if license_el is not None and license_el.text else None

        if arxiv_id and title and abstract:
            yield Paper(
                arxiv_id=arxiv_id,
                title=" ".join(title.split()),      # collapse arXiv's newlines
                abstract=" ".join(abstract.split()),
                authors=authors,
                categories=categories,
                published=published,
                license=license_url,
            )


def fetch_papers(max_papers: int = config.MAX_PAPERS) -> list[Paper]:
    """Download up to `max_papers` papers from arXiv, paging politely.

    Sleeps between pages to respect arXiv's rate guidance — hammering the API
    risks a temporary block, which would make the pipeline *less* reproducible,
    not more. De-duplicates by arXiv id in case paging overlaps.
    """
    papers: dict[str, Paper] = {}
    start = 0
    while len(papers) < max_papers:
        url = _build_query_url(start, config.ARXIV_PAGE_SIZE)
        with urllib.request.urlopen(url, timeout=60) as resp:  # noqa: S310 (trusted host)
            xml_bytes = resp.read()

        page = list(_parse_feed(xml_bytes))
        if not page:
            break  # ran out of results before hitting max_papers
        for p in page:
            papers[p.arxiv_id] = p

        start += config.ARXIV_PAGE_SIZE
        time.sleep(config.ARXIV_REQUEST_DELAY_SECONDS)

    return list(papers.values())[:max_papers]


def fetch_by_ids(arxiv_ids: tuple[str, ...]) -> list[Paper]:
    """Fetch specific papers by arXiv id via the API's id_list parameter.

    Used to pull the seed set of seminal papers so the evaluation's ground truth
    is always present. One request covers all ids.
    """
    if not arxiv_ids:
        return []
    query = f"{ARXIV_API_URL}?id_list={','.join(arxiv_ids)}&max_results={len(arxiv_ids)}"
    with urllib.request.urlopen(query, timeout=60) as resp:  # noqa: S310 (trusted host)
        xml_bytes = resp.read()
    time.sleep(config.ARXIV_REQUEST_DELAY_SECONDS)
    return list(_parse_feed(xml_bytes))


def _sha256(path) -> str:
    """SHA-256 of a file's bytes — the provenance fingerprint."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def save_papers(papers: list[Paper], path=None, provenance_path=None) -> None:
    """Write papers to JSONL and record provenance alongside them.

    The paths are parameters (defaulting to the main corpus) so that any *other*
    slice of arXiv this project downloads — currently the drift probe's newer
    slice, see `drift.py` — gets the same provenance treatment: same fields, same
    SHA-256 fingerprint, same code. A second downloader with its own weaker
    provenance is exactly how a corpus quietly becomes untraceable.
    """
    path = path or config.PAPERS_FILE
    provenance_path = provenance_path or config.PROVENANCE_FILE

    config.ensure_dirs()
    with open(path, "w", encoding="utf-8") as f:
        for p in papers:
            f.write(json.dumps(asdict(p), ensure_ascii=False) + "\n")

    provenance = {
        "source": "arXiv API (http://export.arxiv.org/api/query)",
        "categories": list(config.ARXIV_CATEGORIES),
        "downloaded_utc": datetime.now(timezone.utc).isoformat(),
        "paper_count": len(papers),
        "redistributable_count": sum(p.redistributable for p in papers),
        "papers_file": path.name,
        "papers_sha256": _sha256(path),
    }
    with open(provenance_path, "w", encoding="utf-8") as f:
        json.dump(provenance, f, indent=2)


def load_papers(path=None) -> list[Paper]:
    """Load papers previously saved by `save_papers`."""
    papers: list[Paper] = []
    with open(path or config.PAPERS_FILE, encoding="utf-8") as f:
        for line in f:
            papers.append(Paper(**json.loads(line)))
    return papers


def acquire() -> list[Paper]:
    """Full acquisition step: fetch recent + seed papers, save, return.

    The seed papers (seminal works the evaluation asks about) are merged in and
    de-duplicated by id so they are always present regardless of the recent slice.
    Called by run_pipeline.py.
    """
    recent = fetch_papers()
    seed = fetch_by_ids(config.SEED_PAPER_IDS)
    merged: dict[str, Paper] = {p.arxiv_id: p for p in recent}
    for p in seed:                      # seed papers win on id collision
        merged[p.arxiv_id] = p
    papers = list(merged.values())
    save_papers(papers)
    return papers
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.acquisition'] = _m
setattr(_src, 'acquisition', _m)
exec(compile(_code_acquisition, 'src/acquisition.py', 'exec'), _m.__dict__)

# ---- src/llm.py -----------------------------------------------------
_m = types.ModuleType('src.llm')
_m.__dict__['__name__'] = 'src.llm'
_m.__dict__['__file__'] = _ROOT + '/src/llm.py'
_m.__dict__['__package__'] = 'src'
_code_llm = r'''"""Claude wrapper with an on-disk response cache — the project's LLM boundary.

Everything that calls the Anthropic API goes through here. Two jobs:

1. **Cache every response by a hash of its inputs.** The cache is committed to
   `outputs/results/llm_cache.json`, so the notebook, the tests, and
   `test_pipeline_integrity.py` reproduce every published number *without an API
   key* — exactly the "reference-only, load the saved result" convention the
   portfolio's Kaggle-GPU notebooks use. Re-running the pipeline also never pays
   for an identical call twice.

2. **Degrade gracefully with no key.** If `ANTHROPIC_API_KEY` is unset and a call
   isn't already cached, we raise a clear, catchable error rather than a confusing
   SDK exception — the caller can then decide to skip that step.

The model id, token cap, and cache path are all named constants in config.py.
Uses the official `anthropic` SDK (not raw HTTP) with adaptive thinking, per the
current Claude API guidance.
"""

from __future__ import annotations

import functools
import hashlib
import json
from typing import Any

from src import config


class LLMKeyMissing(RuntimeError):
    """Raised when a live call is needed but no API key is available.

    Distinct exception type so callers (the eval harness, ask.py) can catch it
    and fall back to "reproduce from cache only" instead of crashing.
    """


# --------------------------------------------------------------------------- #
# Cache
# --------------------------------------------------------------------------- #
@functools.lru_cache(maxsize=1)
def _load_cache() -> dict[str, Any]:
    """Load the on-disk response cache (once per process)."""
    if config.LLM_CACHE_FILE.exists():
        with open(config.LLM_CACHE_FILE, encoding="utf-8") as f:
            return json.load(f)
    return {}


def _save_cache(cache: dict[str, Any]) -> None:
    """Persist the response cache."""
    config.ensure_dirs()
    with open(config.LLM_CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, ensure_ascii=False)


def _cache_key(payload: dict[str, Any]) -> str:
    """Deterministic hash of the request payload.

    Sorting keys makes the hash independent of dict ordering, so the same logical
    request always maps to the same cache entry.
    """
    blob = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()


# --------------------------------------------------------------------------- #
# Client
# --------------------------------------------------------------------------- #
@functools.lru_cache(maxsize=1)
def _get_client():
    """Construct the Anthropic client (lazy — only when a live call happens)."""
    import anthropic

    return anthropic.Anthropic()


def _blocks_to_dicts(content) -> list[dict[str, Any]]:
    """Serialize Claude response content blocks to plain dicts for caching."""
    out: list[dict[str, Any]] = []
    for block in content:
        if block.type == "text":
            out.append({"type": "text", "text": block.text})
        elif block.type == "tool_use":
            out.append(
                {"type": "tool_use", "id": block.id, "name": block.name, "input": block.input}
            )
        # thinking blocks carry no content under the default display; skip them.
    return out


def call(
    messages: list[dict[str, Any]],
    system: str | None = None,
    tools: list[dict[str, Any]] | None = None,
    model: str = config.GEN_MODEL,
    max_tokens: int = config.MAX_ANSWER_TOKENS,
    use_cache: bool = True,
) -> dict[str, Any]:
    """Call Claude once and return a plain-dict response.

    The return shape is `{"stop_reason": str, "content": [blocks...]}` where each
    block is a serialized dict — everything the agent loop and the answer parser
    need, and everything that round-trips cleanly through the JSON cache.

    On a cache hit we never touch the network (so no key is needed). On a miss
    with no key we raise `LLMKeyMissing` so the caller can fall back.
    """
    payload = {
        "model": model,
        "max_tokens": max_tokens,
        "system": system,
        "messages": messages,
        "tools": tools,
    }
    key = _cache_key(payload)

    if use_cache:
        cache = _load_cache()
        if key in cache:
            return cache[key]

    if not config.api_key_available():
        raise LLMKeyMissing(
            "No ANTHROPIC_API_KEY set and this request is not cached. "
            "Set the key to generate it, or run only the cached/local steps."
        )

    client = _get_client()
    request: dict[str, Any] = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        # Adaptive thinking: Claude decides how much to reason per request.
        "thinking": {"type": "adaptive"},
    }
    if system is not None:
        request["system"] = system
    if tools is not None:
        request["tools"] = tools

    response = client.messages.create(**request)
    result = {
        "stop_reason": response.stop_reason,
        "content": _blocks_to_dicts(response.content),
    }

    if use_cache:
        cache = _load_cache()
        cache[key] = result
        _save_cache(cache)
        _load_cache.cache_clear()  # so the next read sees the write

    return result


def first_text(result: dict[str, Any]) -> str:
    """Extract the concatenated text of a response (ignoring tool_use blocks)."""
    return "".join(b["text"] for b in result["content"] if b["type"] == "text").strip()
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.llm'] = _m
setattr(_src, 'llm', _m)
exec(compile(_code_llm, 'src/llm.py', 'exec'), _m.__dict__)

# ---- src/embeddings.py ----------------------------------------------
_m = types.ModuleType('src.embeddings')
_m.__dict__['__name__'] = 'src.embeddings'
_m.__dict__['__file__'] = _ROOT + '/src/embeddings.py'
_m.__dict__['__package__'] = 'src'
_code_embeddings = r'''"""Turn text into vectors with a local sentence-transformer.

An *embedding* is the whole game behind semantic retrieval: it maps a piece of
text to a point in a high-dimensional space such that texts about similar things
land near each other. "How do transformers attend to tokens?" and "the attention
mechanism computes weighted sums over the sequence" have almost no words in
common, but their embeddings are close — so a query can find a chunk by *meaning*,
not by keyword overlap.

We use `all-MiniLM-L6-v2` (chosen and justified in config.py). It runs on CPU,
needs no API key, and is deterministic given a fixed seed — which is what keeps
the entire retrieval path reproducible on any machine.

The model is loaded lazily and cached: importing this module is cheap, and the
(~90 MB) model downloads only the first time it is actually used.
"""

from __future__ import annotations

import functools

import numpy as np

from src import config


@functools.lru_cache(maxsize=1)
def get_model():
    """Load (and cache) the sentence-transformer.

    Import is inside the function so that merely importing this module — which
    the tests and the config-reading tools do — does not require
    sentence-transformers to be installed or the model to be downloaded.
    """
    from sentence_transformers import SentenceTransformer

    return SentenceTransformer(config.EMBED_MODEL)


def embed_texts(texts: list[str]) -> np.ndarray:
    """Embed a list of texts into an (n, EMBED_DIM) float32 array.

    Vectors are L2-normalized so that an inner product between two of them equals
    their cosine similarity. That equivalence is what lets the FAISS index use a
    fast inner-product search and have it mean "most semantically similar" —
    see vectorstore.py.
    """
    model = get_model()
    vectors = model.encode(
        texts,
        batch_size=config.EMBED_BATCH_SIZE,
        normalize_embeddings=True,      # unit vectors → inner product = cosine
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    return vectors.astype(np.float32)


def embed_query(query: str) -> np.ndarray:
    """Embed a single query into a (1, EMBED_DIM) array, ready for search."""
    return embed_texts([query])
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.embeddings'] = _m
setattr(_src, 'embeddings', _m)
exec(compile(_code_embeddings, 'src/embeddings.py', 'exec'), _m.__dict__)

# ---- src/vectorstore.py ---------------------------------------------
_m = types.ModuleType('src.vectorstore')
_m.__dict__['__name__'] = 'src.vectorstore'
_m.__dict__['__file__'] = _ROOT + '/src/vectorstore.py'
_m.__dict__['__package__'] = 'src'
_code_vectorstore = r'''"""The FAISS vector store — build, save, load, and search the index.

This is the single definition of "search the corpus" in the whole project. Both
the plain RAG pipeline (`rag.py`) and the agentic system (`agent.py`) call
`VectorStore.search`; neither re-implements retrieval. That is deliberate — if
there were two search paths they could disagree, and no published number could be
trusted. (Same "single source of truth" rule the rest of the portfolio applies to
its core logic.)

Why FAISS? It is an in-process, file-backed nearest-neighbour index — no server,
no Docker, no network. The whole thing runs on a laptop, which is what keeps the
retrieval path reproducible top-to-bottom in the capstone notebook.

Why an inner-product index? Our embeddings are L2-normalized (see embeddings.py),
so inner product == cosine similarity. `IndexFlatIP` does an *exact* search — it
compares the query against every chunk. For a few thousand chunks that is
instant, and it removes approximate-search recall as a variable, so retrieval
quality reflects the embeddings and chunking, not an ANN approximation.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

from src import config
from src.embeddings import embed_query, embed_texts


@dataclass
class Retrieved:
    """One search hit: a chunk, its citation metadata, and its similarity score."""

    chunk_id: str
    arxiv_id: str
    title: str
    text: str
    score: float        # cosine similarity in [-1, 1]; higher is more similar


class VectorStore:
    """An in-memory FAISS index plus the row-aligned chunk metadata.

    The index stores only vectors; the metadata frame stores the text and
    citation fields. They are kept in the same row order, so FAISS returning row
    `i` means "chunk at metadata row `i`". Keeping them aligned is the one
    invariant this class protects.
    """

    def __init__(self, index, meta: pd.DataFrame):
        self._index = index
        self._meta = meta.reset_index(drop=True)

    # ---- construction ---------------------------------------------------- #
    @classmethod
    def build(cls, chunks: pd.DataFrame) -> "VectorStore":
        """Embed every chunk and build the index. This is the expensive step."""
        import faiss

        vectors = embed_texts(chunks["text"].tolist())
        index = faiss.IndexFlatIP(config.EMBED_DIM)
        index.add(vectors)
        # Store exactly the columns needed to return and cite a hit.
        meta = chunks[["chunk_id", "arxiv_id", "title", "text"]].copy()
        return cls(index, meta)

    # ---- persistence ----------------------------------------------------- #
    def save(self) -> None:
        """Persist the index and metadata to `data/index/`."""
        import faiss

        config.ensure_dirs()
        faiss.write_index(self._index, str(config.FAISS_INDEX_FILE))
        self._meta.to_parquet(config.CHUNK_META_FILE, index=False)

    @classmethod
    def load(cls) -> "VectorStore":
        """Load a previously-saved index. Fast — no embedding required."""
        import faiss

        index = faiss.read_index(str(config.FAISS_INDEX_FILE))
        meta = pd.read_parquet(config.CHUNK_META_FILE)
        return cls(index, meta)

    # ---- search ---------------------------------------------------------- #
    @property
    def size(self) -> int:
        """Number of chunks in the index."""
        return int(self._index.ntotal)

    @property
    def meta(self) -> pd.DataFrame:
        """The row-aligned chunk metadata.

        Exposed so the benchmark's other retrieval arms (`retrievers.py`) can
        index the *same* chunks in the *same* row order as this index. Sharing
        the frame is what lets a lexical ranking and a dense ranking be fused by
        row id without a join — and what guarantees the arms are compared on an
        identical collection rather than two slightly different ones.
        """
        return self._meta

    def search_rows(self, query: str, k: int = config.TOP_K) -> list[tuple[int, float]]:
        """Return `(row, score)` pairs for the `k` nearest chunks, best first.

        The raw ranking, before metadata is attached. `search` builds on this,
        and so does reciprocal-rank fusion — which needs positions and row ids,
        not citation fields. Keeping both on top of one method means there is
        still exactly one place where this project asks FAISS a question.

        FAISS returns -1 for empty slots when `k` exceeds the index size; those
        are skipped rather than returned as rows.
        """
        q = embed_query(query)
        scores, idxs = self._index.search(q, k)
        return [
            (int(row), float(score))
            for score, row in zip(scores[0], idxs[0])
            if row >= 0
        ]

    def search(self, query: str, k: int = config.TOP_K) -> list[Retrieved]:
        """Return the `k` chunks most similar to `query`, best first.

        Embed the query, ask FAISS for the top-k by inner product, then map each
        returned row back to its chunk metadata.
        """
        hits: list[Retrieved] = []
        for row, score in self.search_rows(query, k=k):
            m = self._meta.iloc[row]
            hits.append(
                Retrieved(
                    chunk_id=m["chunk_id"],
                    arxiv_id=m["arxiv_id"],
                    title=m["title"],
                    text=m["text"],
                    score=float(score),
                )
            )
        return hits
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.vectorstore'] = _m
setattr(_src, 'vectorstore', _m)
exec(compile(_code_vectorstore, 'src/vectorstore.py', 'exec'), _m.__dict__)

# ---- src/rag.py -----------------------------------------------------
_m = types.ModuleType('src.rag')
_m.__dict__['__name__'] = 'src.rag'
_m.__dict__['__file__'] = _ROOT + '/src/rag.py'
_m.__dict__['__package__'] = 'src'
_code_rag = r'''"""Naive (single-shot) RAG — the baseline the agent has to beat.

The classic RAG pipeline, in three moves:

    retrieve once  ->  stuff the chunks into a grounded prompt  ->  generate

It retrieves the top-k chunks for the question, hands them to Claude with strict
instructions to answer *only* from those chunks and cite them, and returns the
answer plus the sources. There is no iteration: one retrieval, one generation.

This is the honest baseline. Its weakness is built in — if the single retrieval
misses a fact (common on multi-hop questions that need two different pieces of
evidence), the model never gets a second chance. The agentic version in agent.py
exists precisely to fix that, and the evaluation measures whether it does.
"""

from __future__ import annotations

from dataclasses import dataclass, field

from src import config, llm
from src.vectorstore import Retrieved, VectorStore

# The grounding contract. Two rules do the real work: answer ONLY from the
# provided context (this is what stops the model from falling back on its
# parametric memory and hallucinating), and cite the arXiv ids you used (this is
# what makes the answer auditable). Telling it to say when the context is
# insufficient is what lets "I don't know" be a correct answer instead of a
# confident wrong one.
SYSTEM_PROMPT = (
    "You are a research assistant answering questions about machine-learning "
    "papers. Answer using ONLY the provided context passages. Every claim must be "
    "grounded in the passages — do not use outside knowledge. Cite the arXiv ids "
    "of the passages you used, in square brackets, e.g. [2401.01234]. If the "
    "passages do not contain enough information to answer, say so plainly rather "
    "than guessing."
)


@dataclass
class RAGAnswer:
    """A grounded answer plus everything needed to evaluate and audit it."""

    question: str
    answer: str
    sources: list[Retrieved]
    cited_ids: list[str] = field(default_factory=list)


def format_context(hits: list[Retrieved]) -> str:
    """Render retrieved chunks into a numbered, id-tagged context block.

    Each passage is labelled with its arXiv id so the model can cite it and a
    reader can trace every claim back to a specific chunk.
    """
    return "\n\n".join(
        f"[{h.arxiv_id}] {h.title}\n{h.text}" for h in hits
    )


def extract_cited_ids(answer: str, hits: list[Retrieved]) -> list[str]:
    """Find which retrieved arXiv ids the answer actually cited.

    We only count ids that were in the retrieved set — a "citation" of something
    that wasn't retrieved is a hallucinated reference, not a real source.
    """
    retrieved_ids = {h.arxiv_id for h in hits}
    return [aid for aid in retrieved_ids if f"[{aid}]" in answer]


def answer(store: VectorStore, question: str, k: int = config.TOP_K) -> RAGAnswer:
    """Run naive RAG end to end and return a grounded, cited answer."""
    hits = store.search(question, k=k)
    context = format_context(hits)
    user = (
        f"Context passages:\n\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer using only the passages above, and cite the arXiv ids you used."
    )
    result = llm.call(
        messages=[{"role": "user", "content": user}],
        system=SYSTEM_PROMPT,
    )
    text = llm.first_text(result)
    return RAGAnswer(
        question=question,
        answer=text,
        sources=hits,
        cited_ids=extract_cited_ids(text, hits),
    )
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.rag'] = _m
setattr(_src, 'rag', _m)
exec(compile(_code_rag, 'src/rag.py', 'exec'), _m.__dict__)

# ---- src/agent.py ---------------------------------------------------
_m = types.ModuleType('src.agent')
_m.__dict__['__name__'] = 'src.agent'
_m.__dict__['__file__'] = _ROOT + '/src/agent.py'
_m.__dict__['__package__'] = 'src'
_code_agent = r'''"""Agentic RAG — a from-scratch Claude tool-use loop over the retriever.

What makes this *agentic* rather than just RAG? Naive RAG (rag.py) retrieves once,
up front, and hopes that single search grabbed everything. This agent instead
*decides* how to retrieve: it can break a question into parts, search for each,
read what came back, judge whether it has enough, and search again — before it
answers. It runs a real Claude tool-use loop:

    Claude, given a `search_corpus` tool, chooses a query
      -> we run the retrieval and hand back the chunks
      -> Claude reads them, decides: enough? if not, search again (new query)
      -> ... up to MAX_AGENT_STEPS times ...
      -> Claude writes the final grounded, cited answer

We implement the loop by hand — not with a framework — because explainability is
a hard requirement here: every decision the agent makes is a visible message we
log, so the notebook can show its reasoning trace. It uses the same VectorStore
as naive RAG (one definition of retrieval in the repo), so any quality difference
between the two is about the *loop*, not about a different search.

The pay-off this is built to demonstrate: on multi-hop questions — the kind that
need two different pieces of evidence — a single retrieval usually misses one of
them, and the agent's second, reformulated search is what recovers it.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any

from src import config, llm
from src.rag import extract_cited_ids
from src.vectorstore import Retrieved, VectorStore

# The agent's guidance. The key instructions: decompose multi-part questions and
# search for each part separately (this is what beats single-shot retrieval),
# only answer once you have the evidence, ground every claim, and cite arXiv ids.
SYSTEM_PROMPT = (
    "You are a research assistant answering questions about machine-learning "
    "papers. You have a `search_corpus` tool that returns passages from a corpus "
    "of arXiv papers. Strategy: if a question has multiple parts or needs more "
    "than one fact, search for each part separately rather than in one broad "
    "query — a focused query retrieves better. Read what each search returns "
    "before deciding whether you need another. When you have enough evidence, "
    "write the answer using ONLY the retrieved passages; do not use outside "
    "knowledge. Cite the arXiv ids you used in square brackets, e.g. [2401.01234]. "
    "If the corpus cannot answer the question, say so plainly."
)

# The one tool the agent may call. A tight schema — one string query — keeps the
# agent's choices legible and the loop easy to reason about.
SEARCH_TOOL = {
    "name": "search_corpus",
    "description": (
        "Search the arXiv paper corpus for passages relevant to a query. "
        "Returns the most semantically similar passages with their arXiv ids. "
        "Call this multiple times with different focused queries to gather "
        "evidence for different parts of a question."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "A focused search query for one fact or sub-question.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}


@dataclass
class AgentStep:
    """One retrieval the agent chose to make — the auditable reasoning trace."""

    query: str
    hits: list[Retrieved]


@dataclass
class AgentAnswer:
    """The agent's final answer plus its full search trace.

    `steps` is what makes the agent explainable: every query it decided to run and
    what came back, in order. `sources` is the de-duplicated union of everything
    retrieved across all steps — the evidence pool the answer was grounded in.
    """

    question: str
    answer: str
    steps: list[AgentStep] = field(default_factory=list)
    sources: list[Retrieved] = field(default_factory=list)
    cited_ids: list[str] = field(default_factory=list)
    stopped_reason: str = "answered"  # "answered" or "max_steps"


def _format_tool_result(hits: list[Retrieved]) -> str:
    """Render one search's hits into the text handed back to the agent."""
    if not hits:
        return "No passages found for that query."
    return "\n\n".join(f"[{h.arxiv_id}] {h.title}\n{h.text}" for h in hits)


def answer(store: VectorStore, question: str, k: int = config.TOP_K) -> AgentAnswer:
    """Run the agentic loop and return the answer plus its search trace.

    This is a manual Claude tool-use loop: we call the model, and while it keeps
    asking to use the search tool we run the search and feed the results back,
    appending every turn to the running message history. The loop is bounded by
    MAX_AGENT_STEPS so a confused agent can't retrieve forever — after the cap we
    ask it once more, tools disabled, to answer with what it has.
    """
    messages: list[dict[str, Any]] = [{"role": "user", "content": question}]
    steps: list[AgentStep] = []
    all_hits: dict[str, Retrieved] = {}   # de-dupe the evidence pool by chunk_id

    for _ in range(config.MAX_AGENT_STEPS):
        # Inside the loop the search tool is always offered; the model itself
        # decides whether to search again or answer. The step cap below is the
        # backstop if it never chooses to stop searching.
        result = llm.call(messages=messages, system=SYSTEM_PROMPT, tools=[SEARCH_TOOL])

        # Record the assistant turn verbatim so the tool-use blocks are preserved
        # for the next request (the API needs them to match the tool results).
        messages.append({"role": "assistant", "content": result["content"]})

        tool_uses = [b for b in result["content"] if b["type"] == "tool_use"]
        if not tool_uses:
            # No tool call -> the agent chose to answer. We're done.
            text = llm.first_text(result)
            sources = list(all_hits.values())
            return AgentAnswer(
                question=question,
                answer=text,
                steps=steps,
                sources=sources,
                cited_ids=extract_cited_ids(text, sources),
                stopped_reason="answered",
            )

        # Run every requested search and return all results in one user message
        # (the API expects one tool_result per tool_use, batched together).
        tool_results = []
        for tu in tool_uses:
            query = tu["input"]["query"]
            hits = store.search(query, k=k)
            steps.append(AgentStep(query=query, hits=hits))
            for h in hits:
                all_hits[h.chunk_id] = h
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": tu["id"],
                    "content": _format_tool_result(hits),
                }
            )
        messages.append({"role": "user", "content": tool_results})

    # Hit the step cap without answering: ask once more with tools off so the
    # model is forced to commit to an answer using the evidence gathered so far.
    messages.append(
        {
            "role": "user",
            "content": (
                "You have gathered enough evidence. Now write the final answer "
                "using only the passages retrieved so far, citing arXiv ids."
            ),
        }
    )
    result = llm.call(messages=messages, system=SYSTEM_PROMPT, tools=None)
    text = llm.first_text(result)
    sources = list(all_hits.values())
    return AgentAnswer(
        question=question,
        answer=text,
        steps=steps,
        sources=sources,
        cited_ids=extract_cited_ids(text, sources),
        stopped_reason="max_steps",
    )
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.agent'] = _m
setattr(_src, 'agent', _m)
exec(compile(_code_agent, 'src/agent.py', 'exec'), _m.__dict__)

# ---- src/chunking.py ------------------------------------------------
_m = types.ModuleType('src.chunking')
_m.__dict__['__name__'] = 'src.chunking'
_m.__dict__['__file__'] = _ROOT + '/src/chunking.py'
_m.__dict__['__package__'] = 'src'
_code_chunking = r'''"""Split documents into overlapping chunks for retrieval.

Why chunk at all? An embedding model turns a piece of text into ONE vector. Feed
it a whole 4,000-word paper and that single vector has to average away every
distinct idea in the paper — retrieval on it is mush. Feed it a few sentences and
the vector actually means something specific, so a query about that specific idea
lands on it. Chunking is what makes semantic retrieval precise.

The two knobs — chunk size and overlap — are named constants in `config.py` with
their rationale. This module is deliberately simple and character-based (no
tokenizer dependency) so the whole retrieval path stays reproducible and easy to
reason about; every chunk carries its parent paper's metadata so an answer can
cite the paper it came from.
"""

from __future__ import annotations

from dataclasses import dataclass

import pandas as pd

from src import config
from src.acquisition import Paper


@dataclass
class Chunk:
    """One retrievable unit of text plus the metadata needed to cite it."""

    chunk_id: str        # e.g. "2401.01234:2" — paper id + position
    arxiv_id: str
    title: str
    text: str
    position: int        # 0-based index of this chunk within its paper


def split_text(text: str,
               size: int = config.CHUNK_SIZE_CHARS,
               overlap: int = config.CHUNK_OVERLAP_CHARS) -> list[str]:
    """Slice `text` into overlapping windows of `size` characters.

    Each window starts `size - overlap` characters after the previous one, so
    consecutive chunks share `overlap` characters. That shared margin is what
    stops a fact that straddles a boundary from being lost — it survives in at
    least one chunk. We try to end each window at a whitespace boundary so chunks
    don't split mid-word, which would corrupt the embedding.

    Guarding the two edge cases explicitly: overlap must be smaller than size (or
    the window never advances and we loop forever), and text shorter than one
    window is returned as a single chunk.
    """
    if overlap >= size:
        raise ValueError(f"overlap ({overlap}) must be smaller than size ({size})")

    text = text.strip()
    if len(text) <= size:
        return [text] if text else []

    chunks: list[str] = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        # Prefer to break at the last space in the window so we don't cut a word
        # in half — but only if that space isn't so early it shrinks the chunk to
        # nothing (which would happen on long unbroken strings like URLs).
        if end < len(text):
            last_space = text.rfind(" ", start, end)
            if last_space > start + size // 2:
                end = last_space
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        # Advance so the next window overlaps the current one by `overlap` chars.
        start = end - overlap
    return [c for c in chunks if c]


def chunk_paper(paper: Paper,
                size: int = config.CHUNK_SIZE_CHARS,
                overlap: int = config.CHUNK_OVERLAP_CHARS) -> list[Chunk]:
    """Turn one paper into its list of Chunks, carrying metadata through."""
    return [
        Chunk(
            chunk_id=f"{paper.arxiv_id}:{i}",
            arxiv_id=paper.arxiv_id,
            title=paper.title,
            text=piece,
            position=i,
        )
        for i, piece in enumerate(split_text(paper.text, size=size, overlap=overlap))
    ]


def chunk_papers(papers: list[Paper],
                 size: int = config.CHUNK_SIZE_CHARS,
                 overlap: int = config.CHUNK_OVERLAP_CHARS) -> list[Chunk]:
    """Chunk every paper in the corpus.

    `size` and `overlap` default to the shipped constants and are parameters only
    so `run_ablation.py` can rebuild the corpus at other settings and measure what
    the constants are worth. The pipeline never passes them — there is still one
    chunking configuration in the shipped system, defined in `config.py`.
    """
    chunks: list[Chunk] = []
    for p in papers:
        chunks.extend(chunk_paper(p, size=size, overlap=overlap))
    return chunks


def chunks_to_frame(chunks: list[Chunk]) -> pd.DataFrame:
    """Chunks → DataFrame, the on-disk form and the shape the vector store wants."""
    return pd.DataFrame(
        {
            "chunk_id": [c.chunk_id for c in chunks],
            "arxiv_id": [c.arxiv_id for c in chunks],
            "title": [c.title for c in chunks],
            "text": [c.text for c in chunks],
            "position": [c.position for c in chunks],
        }
    )


def save_chunks(chunks: list[Chunk]) -> pd.DataFrame:
    """Chunk the corpus to parquet and return the frame. Called by the pipeline."""
    config.ensure_dirs()
    frame = chunks_to_frame(chunks)
    frame.to_parquet(config.CHUNKS_FILE, index=False)
    return frame


def load_chunks() -> pd.DataFrame:
    """Load the chunk frame written by `save_chunks`."""
    return pd.read_parquet(config.CHUNKS_FILE)
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.chunking'] = _m
setattr(_src, 'chunking', _m)
exec(compile(_code_chunking, 'src/chunking.py', 'exec'), _m.__dict__)

# ---- src/evalset.py -------------------------------------------------
_m = types.ModuleType('src.evalset')
_m.__dict__['__name__'] = 'src.evalset'
_m.__dict__['__file__'] = _ROOT + '/src/evalset.py'
_m.__dict__['__package__'] = 'src'
_code_evalset = r'''"""Generate the known-item evaluation set — 120 graded queries, no API key.

The original evaluation set is 11 hand-written questions (`eval/qa_set.json`).
They are good questions, and they stay. But 11 is too few to put a confidence
interval on: a single question flipping moves the mean by nine points, so any
"arm A beats arm B" claim over 11 rows is noise wearing a decimal point.

This module builds a larger set the only way it honestly can without a human
adjudicator or an API key: **known-item retrieval**. The construction is:

    take a paper -> derive a query from it -> the right answer is that paper

This is a standard information-retrieval evaluation design (it is how much of
the TREC known-item and BEIR-style corpora are built): the relevance judgement
is not a matter of opinion, because the query was *manufactured from* the
document it points to. That gives 120 unambiguous labels for free.

A known-item set built one way would be a single, arguable data point. So every
sampled paper yields **three queries, in three styles**, reported separately:

  * **abstract** — the longest sentence from the abstract. Fluent natural prose,
    and a verbatim substring of the passage it targets. The *easiest* possible
    query, and it turns out every arm solves it.
  * **title** — the paper's title. Still natural language, still literal overlap,
    but much shorter and denser in content words.
  * **keyword** — five content words sampled from across the abstract, stop-words
    removed and word order shuffled. No grammar, not a substring of any single
    passage. This is what people actually type into a search box, and it is the
    furthest thing here from the fluent sentences a sentence-embedding model was
    trained on.

The three styles form a gradient, and comparing one arm across them measures
something no single number can: **how much a retriever's quality depends on the
query being well-formed prose.** BM25 does not read word order at all, so it
should be nearly indifferent. A bi-encoder trained on natural sentences has every
reason not to be. That contrast is the finding, not a caveat — and it is why the
set is generated three ways instead of the one that would have looked tidiest.

`eval/ADJUDICATION.md` writes up what these labels are worth and where they
mislead.

Everything here is seeded with `config.SEED`, so the generated set is byte-identical
on every machine and every run.
"""

from __future__ import annotations

import json
import random
import re

from src import config
from src.acquisition import Paper, load_papers

# A query is one sentence. Splitting on sentence-ending punctuation followed by
# whitespace and a capital letter is crude, but abstracts are well-punctuated
# prose and this avoids a heavyweight NLP dependency for a job that does not need
# one. Over-splitting is self-correcting: fragments below MIN_QUERY_CHARS are
# dropped anyway.
_SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z])")

# Sentences that begin like these identify the paper's *housekeeping*, not its
# content — every paper has them, so they point at no particular document and
# would be unanswerable by any retriever. Dropping them keeps the ground truth
# meaningful rather than making the benchmark artificially hard for everyone.
_BOILERPLATE_PREFIXES = (
    "we release", "our code", "code is", "code and", "the code",
    "data and", "we make", "all code", "we publicly", "we open",
    "in this paper", "in this work",
)

# Words too common to identify a paper. Used only to build the *keyword* query
# style — never by the retrievers themselves, where BM25's IDF already handles
# this and a hand-curated stop list would be an untunable thumb on the scale.
# Includes the academic-writing filler ("propose", "novel", "results") that
# appears in nearly every abstract in the corpus.
_STOPWORDS = frozenset("""
a an and are as at be been between both by can each for from had has have if in
into is it its may more most not of on or other our over paper propose proposed
result results show shows study such than that the their then these this those
three to two under use used using via was we were when where which while with
approach method methods new novel work
""".split())

# A content word: letters and hyphens, at least three characters. Numbers and
# single letters are dropped — "2" and "x" identify nothing.
_WORD = re.compile(r"[A-Za-z][A-Za-z\-]{2,}")


def _sentences(abstract: str) -> list[str]:
    """Split an abstract into candidate query sentences, longest-first.

    Longest-first because a longer sentence carries more of the paper's specific
    content, and specificity is exactly what makes a known-item query solvable.
    """
    parts = [s.strip() for s in _SENTENCE_SPLIT.split(abstract) if s.strip()]
    keep = [
        s for s in parts
        if len(s) >= config.MIN_QUERY_CHARS
        and not s.lower().startswith(_BOILERPLATE_PREFIXES)
    ]
    return sorted(keep, key=len, reverse=True)


def _keyword_query(abstract: str, seed: int) -> str:
    """A short, unordered bag of content words drawn from the abstract.

    Deliberately *not* a sentence. Word order is shuffled and grammar is gone, so
    this is the query style furthest from the natural prose a sentence-embedding
    model was trained on — and closest to what people actually type into a search
    box. Seeded per paper, so the same paper always yields the same query.

    Sampling from the whole abstract rather than one sentence also means the
    query is not a substring of any single passage, which is what stops it from
    being a verbatim-match exercise.
    """
    rng = random.Random(seed)
    words: list[str] = []
    for w in _WORD.findall(abstract.lower()):
        if w not in _STOPWORDS and w not in words:
            words.append(w)
    if len(words) > config.KEYWORD_QUERY_WORDS:
        words = rng.sample(words, config.KEYWORD_QUERY_WORDS)
    rng.shuffle(words)
    return " ".join(words)


def _eligible(paper: Paper) -> bool:
    """True when a paper can yield both query flavours.

    A paper needs a usable title and at least one substantial, non-boilerplate
    abstract sentence. Requiring *both* flavours from every sampled paper keeps
    the two slices perfectly paired — same papers, same difficulty, only the
    query style differs — which is what makes the title-vs-abstract comparison a
    controlled one rather than two unrelated samples.
    """
    return len(paper.title) >= config.MIN_QUERY_CHARS and bool(_sentences(paper.abstract))


def build_known_item_set(papers: list[Paper] | None = None) -> list[dict]:
    """Build the known-item query set. Deterministic under `config.SEED`.

    Returns one dict per query, in the same shape the hand-written
    `eval/qa_set.json` uses (`qid`, `question`, `relevant_ids`, `hop`) so the
    existing metric code consumes both sets without a special case — plus a
    `flavour` field the benchmark slices on.

    Every eligible paper contributes one query in each of the three styles, so
    the slices are perfectly paired: same papers, same targets, only the query
    style differs. That pairing is what makes comparing an arm across styles a
    controlled measurement rather than three unrelated samples.

    `hop` is "single" for every generated query: a known-item query is answered
    by exactly one paper, by construction. Multi-hop questions cannot be
    manufactured this way, which is precisely why the 11 hand-written ones
    remain the harder slice and are reported separately.
    """
    if papers is None:
        papers = load_papers()

    eligible = sorted((p for p in papers if _eligible(p)), key=lambda p: p.arxiv_id)
    # Sort before sampling: `load_papers` returns whatever order the JSONL was
    # written in, and a seeded sample over an unsorted list is only reproducible
    # if the input order is too. Sorting by id pins it.

    rng = random.Random(config.SEED)
    n = min(config.KNOWN_ITEM_PAPERS, len(eligible))
    sampled = rng.sample(eligible, n)

    rows: list[dict] = []
    for i, paper in enumerate(sampled):
        rows.append(
            {
                "qid": f"kt{i:03d}",
                "question": paper.title,
                "relevant_ids": [paper.arxiv_id],
                "hop": "single",
                "flavour": "title",
            }
        )
        rows.append(
            {
                "qid": f"ka{i:03d}",
                "question": _sentences(paper.abstract)[0],
                "relevant_ids": [paper.arxiv_id],
                "hop": "single",
                "flavour": "abstract",
            }
        )
        rows.append(
            {
                "qid": f"kk{i:03d}",
                "question": _keyword_query(paper.abstract, config.SEED + i),
                "relevant_ids": [paper.arxiv_id],
                "hop": "single",
                "flavour": "keyword",
            }
        )
    return rows


def save_known_item_set(rows: list[dict]) -> None:
    """Write the generated set to `eval/known_item_set.json`.

    This file *is* committed, unlike the corpus. It is small, and every published
    number depends on it — a reviewer who wants to check a claim needs to see the
    exact queries it was measured on, not a promise that they were regenerable.
    """
    config.EVAL_DIR.mkdir(parents=True, exist_ok=True)
    with open(config.KNOWN_ITEM_FILE, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2, ensure_ascii=False)


def load_known_item_set() -> list[dict]:
    """Load the committed known-item set, generating it if absent."""
    if not config.KNOWN_ITEM_FILE.exists():
        rows = build_known_item_set()
        save_known_item_set(rows)
        return rows
    with open(config.KNOWN_ITEM_FILE, encoding="utf-8") as f:
        return json.load(f)
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.evalset'] = _m
setattr(_src, 'evalset', _m)
exec(compile(_code_evalset, 'src/evalset.py', 'exec'), _m.__dict__)

# ---- src/evaluation.py ----------------------------------------------
_m = types.ModuleType('src.evaluation')
_m.__dict__['__name__'] = 'src.evaluation'
_m.__dict__['__file__'] = _ROOT + '/src/evaluation.py'
_m.__dict__['__package__'] = 'src'
_code_evaluation = r'''"""Evaluation — does the agent actually beat naive RAG, and by how much?

This is the methodology track that carries the project. It has three parts, and
the first two need no API key:

1. **Retrieval metrics (fully local, no API key).** For each evaluation question
   we know which paper(s) genuinely answer it. We measure whether retrieval
   surfaced them:
     - hit@k   : did *any* relevant paper appear in the top-k?
     - recall@k: what fraction of the relevant papers appeared?
     - MRR     : 1 / rank of the first relevant paper (rewards ranking it high).
   These use only the vector store, so they reproduce exactly on any machine.

2. **The retrieval benchmark (fully local, no API key).** The four arms in
   `retrievers.py` — dense, BM25, hybrid RRF, cross-encoder rerank — scored on
   the same questions with nDCG@10, MAP, recall@k and MRR, each with a bootstrap
   confidence interval, plus per-arm latency. This is what turns "we built a RAG
   system" into "we measured four of them and here is what each one buys and
   costs". The question set it runs on is 120 generated known-item queries
   (`evalset.py`) alongside the 11 hand-written ones; `eval/ADJUDICATION.md`
   states how both sets were labelled and where they mislead.

3. **Answer quality (LLM-as-judge, key-gated + cached).** A separate Claude call
   grades each answer for *groundedness* (is every claim supported by the sources?)
   and *correctness* (does it actually answer the question?), for naive RAG and
   the agent, split by single-hop vs multi-hop. The grades are cached and
   committed, so the headline comparison reproduces from cache without a key.

The design goal is a fair head-to-head: same corpus, same retriever, same
questions — the only thing that differs is single-shot vs iterative retrieval.
"""

from __future__ import annotations

import json
import math
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd

from src import agent as agent_mod
from src import config, llm, rag
from src.vectorstore import VectorStore


# --------------------------------------------------------------------------- #
# Evaluation set
# --------------------------------------------------------------------------- #
@dataclass
class EvalQuestion:
    """One graded question.

    `relevant_ids` are the arXiv ids that genuinely answer it — the ground truth
    for retrieval metrics. `hop` tags the question as needing one fact
    ("single") or several ("multi"); the whole hypothesis is that the agent's
    edge shows up on the multi-hop rows.
    """

    qid: str
    question: str
    relevant_ids: list[str]
    hop: str  # "single" or "multi"
    # Which construction produced this question. "curated" = hand-written by a
    # person (the 11 in qa_set.json); "title" / "abstract" = generated known-item
    # queries (see evalset.py). Defaulted so the hand-written file, which has no
    # such field, still loads unchanged. The benchmark slices on this, because
    # the two generated flavours differ in how much literal word overlap they
    # share with their target — which is the lexical-bias measurement.
    flavour: str = "curated"


def load_eval_set() -> list[EvalQuestion]:
    """Load the curated question set from eval/qa_set.json."""
    with open(config.EVAL_DIR / "qa_set.json", encoding="utf-8") as f:
        raw = json.load(f)
    return [EvalQuestion(**q) for q in raw]


def load_known_item_questions() -> list[EvalQuestion]:
    """Load the 120 generated known-item queries as `EvalQuestion`s."""
    from src.evalset import load_known_item_set

    return [EvalQuestion(**q) for q in load_known_item_set()]


# --------------------------------------------------------------------------- #
# Retrieval metrics (local)
# --------------------------------------------------------------------------- #
def hit_at_k(retrieved_ids: list[str], relevant_ids: list[str]) -> float:
    """1.0 if any relevant id is in the retrieved set, else 0.0."""
    return 1.0 if set(retrieved_ids) & set(relevant_ids) else 0.0


def recall_at_k(retrieved_ids: list[str], relevant_ids: list[str]) -> float:
    """Fraction of the relevant ids that were retrieved."""
    if not relevant_ids:
        return 0.0
    found = set(retrieved_ids) & set(relevant_ids)
    return len(found) / len(set(relevant_ids))


def mrr(retrieved_ids: list[str], relevant_ids: list[str]) -> float:
    """Reciprocal rank of the first relevant id (0 if none retrieved).

    Ranks are 1-based, so a relevant paper in the top slot scores 1.0, second
    slot 0.5, and so on — rewarding retrieval for putting the right paper high,
    not merely somewhere in the list.
    """
    relevant = set(relevant_ids)
    for rank, aid in enumerate(retrieved_ids, start=1):
        if aid in relevant:
            return 1.0 / rank
    return 0.0


def retrieval_ids(store: VectorStore, question: str, k: int = config.TOP_K) -> list[str]:
    """The ordered list of distinct arXiv ids retrieved for a question.

    Chunks are de-duplicated to paper level (keeping first appearance) because
    the ground truth is per-paper: two chunks from the same paper are one hit.
    """
    ids: list[str] = []
    for h in store.search(question, k=k):
        if h.arxiv_id not in ids:
            ids.append(h.arxiv_id)
    return ids


def evaluate_retrieval(store: VectorStore, questions: list[EvalQuestion],
                       k: int = config.TOP_K) -> list[dict]:
    """Compute hit@k, recall@k, and MRR for every question. Fully local."""
    rows = []
    for q in questions:
        got = retrieval_ids(store, q.question, k=k)
        rows.append(
            {
                "qid": q.qid,
                "hop": q.hop,
                "hit@k": hit_at_k(got, q.relevant_ids),
                "recall@k": recall_at_k(got, q.relevant_ids),
                "mrr": mrr(got, q.relevant_ids),
            }
        )
    return rows


# --------------------------------------------------------------------------- #
# Ranking metrics — nDCG and MAP
# --------------------------------------------------------------------------- #
# hit@k, recall@k and MRR above answer "did we find it?" and "was the first one
# early?". Neither notices where the *second* relevant document landed, and MRR
# stops looking after the first hit. The two metrics below grade the whole
# ordering, which is what a retrieval team actually reports.
def dcg(gains: list[float]) -> float:
    """Discounted cumulative gain of a ranked list of relevance gains.

    Each position's gain is divided by log2(rank + 1), so a relevant document at
    rank 1 is worth 1.00, at rank 2 about 0.63, at rank 10 about 0.29. The
    logarithm encodes an assumption worth stating: attention falls off sharply
    with position but never quite to zero — rank 10 is much worse than rank 1,
    not worthless.
    """
    return sum(g / math.log2(i + 1) for i, g in enumerate(gains, start=1))


def ndcg_at_k(retrieved_ids: list[str], relevant_ids: list[str], k: int) -> float:
    """Normalised DCG at k, with binary gains (relevant = 1, else 0).

    Raw DCG is not comparable across questions: a question with three relevant
    papers can score higher than one with a single relevant paper purely because
    there is more to find. Normalising by the *ideal* DCG — the score the perfect
    ranking would have achieved for this question — puts every question on a 0-1
    scale where 1.0 means "nothing could have been ordered better".

    Gains are binary here because the ground truth is binary; see
    `eval/ADJUDICATION.md` for why, and for why that makes these numbers
    incomparable to graded-relevance nDCG from public leaderboards.
    """
    relevant = set(relevant_ids)
    if not relevant:
        return 0.0
    gains = [1.0 if aid in relevant else 0.0 for aid in retrieved_ids[:k]]
    # The ideal ranking puts every relevant document first — but only as many as
    # could fit in k, and only as many as exist.
    ideal = [1.0] * min(len(relevant), k)
    ideal_dcg = dcg(ideal)
    return dcg(gains) / ideal_dcg if ideal_dcg else 0.0


def average_precision(retrieved_ids: list[str], relevant_ids: list[str],
                      k: int) -> float:
    """Average precision at k — the mean of precision measured at each hit.

    Averaged over questions this is MAP, the standard single-number summary in
    the IR literature. It differs from nDCG in what it rewards: AP cares about
    precision *at every relevant document*, so burying the second relevant paper
    at rank 9 hurts it more than nDCG's smooth logarithmic discount does. Both
    are reported because they disagree in an informative way — a large nDCG with
    a weak MAP means the arm reliably finds *one* good document and then loses
    interest.

    Divided by the number of relevant documents (capped at k), not by the number
    found, so failing to retrieve a relevant document is penalised rather than
    ignored.
    """
    relevant = set(relevant_ids)
    if not relevant:
        return 0.0
    hits = 0
    precision_sum = 0.0
    for rank, aid in enumerate(retrieved_ids[:k], start=1):
        if aid in relevant:
            hits += 1
            precision_sum += hits / rank
    denom = min(len(relevant), k)
    return precision_sum / denom if denom else 0.0


# --------------------------------------------------------------------------- #
# Uncertainty — the bootstrap
# --------------------------------------------------------------------------- #
def bootstrap_ci(values: list[float], n_resamples: int = config.BOOTSTRAP_RESAMPLES,
                 confidence: float = config.BOOTSTRAP_CI,
                 seed: int = config.SEED) -> tuple[float, float]:
    """Percentile bootstrap confidence interval for the mean of `values`.

    **Why this is not optional.** A benchmark that reports "arm A scores 0.71,
    arm B scores 0.68" and stops has said nothing: over 120 questions that gap may
    be well inside the noise. The bootstrap answers "if I had drawn a different
    sample of questions, how much would this mean have moved?"

    **What is resampled.** Questions, with replacement — because the question set
    is the sample we are generalising over. The claim being defended is "on
    queries like these", not "on this exact list of 120", and resampling
    questions is what makes the interval mean the former.

    Seeded, so the published interval is reproducible to the digit.
    """
    if not values:
        return (0.0, 0.0)
    rng = np.random.default_rng(seed)
    arr = np.asarray(values, dtype=float)
    # One (n_resamples, n) draw of indices, then a row-wise mean: the whole
    # bootstrap is two array operations, no Python loop.
    idx = rng.integers(0, arr.size, size=(n_resamples, arr.size))
    means = arr[idx].mean(axis=1)
    tail = (1.0 - confidence) / 2.0
    lo, hi = np.quantile(means, [tail, 1.0 - tail])
    return (float(lo), float(hi))


# --------------------------------------------------------------------------- #
# The arm benchmark
# --------------------------------------------------------------------------- #
def _retrieved_paper_ids(retriever, question: str, k: int) -> list[str]:
    """Ordered distinct arXiv ids from one arm's hits.

    The same chunk-to-paper de-duplication `retrieval_ids` applies to the dense
    arm, applied to any arm. Ground truth is per-paper, so two chunks from one
    paper are one hit — and an arm that returns ten chunks of the same paper must
    not be scored as if it found ten relevant documents.
    """
    ids: list[str] = []
    for h in retriever.search(question, k=k):
        if h.arxiv_id not in ids:
            ids.append(h.arxiv_id)
    return ids


def evaluate_arms(retrievers: list, questions: list[EvalQuestion],
                  k: int = config.EVAL_K, warmup: bool = True) -> list[dict]:
    """Score every arm on every question. One row per (arm, question).

    **Latency.** Measured with `perf_counter` around the search call only, after
    a warm-up query per arm. The warm-up matters more than it sounds: the first
    call to the dense arm pays a one-off sentence-transformer load of tens of
    seconds, and the first call to the reranker pays another. Including those in
    the average would not make the reranker look slow — it would make the
    *measurement* wrong, and the whole quality-per-millisecond claim rests on
    this number being honest.

    Latency is still wall-clock on one laptop CPU, single-threaded, batch size
    one. It is a fair *relative* comparison between arms on identical hardware,
    not a throughput figure anyone should plan capacity from.
    """
    if warmup:
        for r in retrievers:
            r.search("warm up the model weights and any lazy imports", k=k)

    rows: list[dict] = []
    for r in retrievers:
        for q in questions:
            start = time.perf_counter()
            got = _retrieved_paper_ids(r, q.question, k=k)
            elapsed_ms = (time.perf_counter() - start) * 1000.0
            rows.append(
                {
                    "arm": r.name,
                    "qid": q.qid,
                    "hop": q.hop,
                    "flavour": q.flavour,
                    "ndcg@k": ndcg_at_k(got, q.relevant_ids, k),
                    "ap@k": average_precision(got, q.relevant_ids, k),
                    "recall@k": recall_at_k(got, q.relevant_ids),
                    "mrr": mrr(got, q.relevant_ids),
                    "hit@k": hit_at_k(got, q.relevant_ids),
                    "latency_ms": elapsed_ms,
                }
            )
    return rows


#: The metrics summarised with a bootstrap interval. Latency is summarised too,
#: but as a median rather than a mean — a single scheduler hiccup can double a
#: mean and say nothing about the arm.
ARM_METRICS = ("ndcg@k", "ap@k", "recall@k", "mrr", "hit@k")


# --------------------------------------------------------------------------- #
# The cutoff sweep — is k = 10 (or k = 5) the right place to look?
# --------------------------------------------------------------------------- #
#: The cutoffs swept. 5 is what the system serves (`config.TOP_K`), 10 is what the
#: benchmark reports (`config.EVAL_K`), and 50 is the reranker's shortlist depth
#: (`config.RERANK_CANDIDATES`) — every constant the project already committed to,
#: plus 1 and 3 to see the top of the curve. Sweeping the numbers you already
#: chose is the cheapest way to find out whether they were choices or habits.
SWEEP_KS = (1, 3, 5, 10, 20, 50)


def sweep_k(retrievers: list, questions: list[EvalQuestion],
            ks: tuple[int, ...] = SWEEP_KS, warmup: bool = True) -> list[dict]:
    """nDCG@k and recall@k for every arm at every cutoff. One row per (arm, k).

    **One search, many cutoffs.** Each arm searches once at `max(ks)` and the
    ranked list is then truncated for each smaller k. Searching separately per
    cutoff would be six times the work *and* less correct: a retriever asked for
    50 and a retriever asked for 5 can return different orderings (the reranker
    demonstrably does — it rescores a shortlist), so the curve would mix "quality
    at cutoff k" with "quality of a retriever configured for k". Truncating one
    ranking isolates the cutoff, which is what the sweep is asking about.

    **What it answers.** Two live questions the fixed-k benchmark cannot:
    where recall saturates — i.e. whether serving `TOP_K = 5` chunks discards
    documents the evaluation at 10 was crediting — and how deep a first stage has
    to go before a reranker has everything it needs, which is the assumption
    `RERANK_CANDIDATES = 50` rests on.
    """
    max_k = max(ks)
    if warmup:
        for r in retrievers:
            r.search("warm up the model weights and any lazy imports", k=max_k)

    rows: list[dict] = []
    for r in retrievers:
        per_k: dict[int, list[tuple[float, float]]] = {k: [] for k in ks}
        for q in questions:
            ranked = _retrieved_paper_ids(r, q.question, max_k)
            for k in ks:
                truncated = ranked[:k]
                per_k[k].append((
                    ndcg_at_k(truncated, q.relevant_ids, k),
                    recall_at_k(truncated, q.relevant_ids),
                ))

        for k, pairs in per_k.items():
            ndcgs = [p[0] for p in pairs]
            recalls = [p[1] for p in pairs]
            lo, hi = bootstrap_ci(ndcgs)
            r_lo, r_hi = bootstrap_ci(recalls)
            rows.append({
                "arm": r.name,
                "k": k,
                "n_questions": len(pairs),
                "ndcg@k": float(np.mean(ndcgs)),
                "ndcg@k_lo": lo,
                "ndcg@k_hi": hi,
                "recall@k": float(np.mean(recalls)),
                "recall@k_lo": r_lo,
                "recall@k_hi": r_hi,
            })
    return rows


def summarise_arms(rows: list[dict], k: int = config.EVAL_K) -> list[dict]:
    """Per-arm, per-slice means with bootstrap intervals and median latency.

    Produces one record per (arm, slice), where slice is "all" plus each question
    flavour present. The flavour split is the point: comparing an arm's `title`
    row against its `abstract` row measures how much it leans on literal word
    overlap (see `eval/ADJUDICATION.md`).
    """
    df = pd.DataFrame(rows)
    out: list[dict] = []
    slices = ["all"] + sorted(df["flavour"].unique().tolist())

    for arm in df["arm"].unique():
        for slice_name in slices:
            sub = df[df.arm == arm]
            if slice_name != "all":
                sub = sub[sub.flavour == slice_name]
            if sub.empty:
                continue
            record = {"arm": arm, "slice": slice_name, "n_questions": int(len(sub)),
                      "k": k}
            for metric in ARM_METRICS:
                values = sub[metric].tolist()
                lo, hi = bootstrap_ci(values)
                record[metric] = float(np.mean(values))
                record[f"{metric}_lo"] = lo
                record[f"{metric}_hi"] = hi
            record["latency_ms_median"] = float(sub["latency_ms"].median())
            record["latency_ms_p95"] = float(sub["latency_ms"].quantile(0.95))
            out.append(record)
    return out


# --------------------------------------------------------------------------- #
# Answer quality (LLM-as-judge)
# --------------------------------------------------------------------------- #
JUDGE_SYSTEM = (
    "You are a strict evaluator of answers to research questions. You are given a "
    "question, a set of source passages, and an answer. Grade two things on a 1-5 "
    "scale:\n"
    "  groundedness: 5 = every claim is directly supported by the passages; "
    "1 = the answer makes claims the passages do not support.\n"
    "  correctness: 5 = fully and accurately answers the question; "
    "1 = wrong or off-topic.\n"
    "Reply with ONLY a JSON object: "
    '{\"groundedness\": <int>, \"correctness\": <int>}.'
)


def judge_answer(question: str, sources_text: str, answer_text: str) -> dict:
    """Grade one answer with the LLM judge; returns groundedness + correctness.

    Cached and key-gated like every other LLM call, so the graded results
    reproduce from the committed cache without an API key.
    """
    user = (
        f"Question: {question}\n\n"
        f"Source passages:\n{sources_text}\n\n"
        f"Answer:\n{answer_text}\n\n"
        "Grade the answer. Reply with only the JSON object."
    )
    result = llm.call(
        messages=[{"role": "user", "content": user}],
        system=JUDGE_SYSTEM,
        model=config.JUDGE_MODEL,
    )
    text = llm.first_text(result)
    try:
        grades = json.loads(text)
        return {
            "groundedness": int(grades["groundedness"]),
            "correctness": int(grades["correctness"]),
        }
    except (json.JSONDecodeError, KeyError, ValueError):
        # A judge that returns unparseable output scores 0 — visible and honest,
        # rather than silently dropping the row.
        return {"groundedness": 0, "correctness": 0}


def evaluate_answers(store: VectorStore, questions: list[EvalQuestion]) -> list[dict]:
    """Run naive RAG and the agent on every question and grade both answers.

    Produces one row per (question, system) so the notebook can compare naive vs
    agent, sliced by hop type. Also records the agent's step count — the token /
    latency cost of its extra retrievals, which is the other side of the trade.
    """
    rows = []
    for q in questions:
        naive = rag.answer(store, q.question)
        agentic = agent_mod.answer(store, q.question)

        naive_sources = rag.format_context(naive.sources)
        agent_sources = rag.format_context(agentic.sources)

        rows.append(
            {
                "qid": q.qid, "hop": q.hop, "system": "naive",
                "n_retrievals": 1,
                "n_cited": len(naive.cited_ids),
                **judge_answer(q.question, naive_sources, naive.answer),
            }
        )
        rows.append(
            {
                "qid": q.qid, "hop": q.hop, "system": "agent",
                "n_retrievals": len(agentic.steps),
                "n_cited": len(agentic.cited_ids),
                **judge_answer(q.question, agent_sources, agentic.answer),
            }
        )
    return rows
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.evaluation'] = _m
setattr(_src, 'evaluation', _m)
exec(compile(_code_evaluation, 'src/evaluation.py', 'exec'), _m.__dict__)

# ---- src/lexical.py -------------------------------------------------
_m = types.ModuleType('src.lexical')
_m.__dict__['__name__'] = 'src.lexical'
_m.__dict__['__file__'] = _ROOT + '/src/lexical.py'
_m.__dict__['__package__'] = 'src'
_code_lexical = r'''"""BM25 — the lexical baseline the dense retriever has to beat.

Dense embedding retrieval is the interesting half of this project, but it is only
interesting if it beats the boring option. BM25 is the boring option: a
term-frequency ranking function from the 1990s, no neural network, no GPU, no
model download. It is still the default first-stage retriever in production
search systems, and it is still competitive on a great many benchmarks. A RAG
system that cannot beat BM25 has not earned its embedding model.

**Why implement it rather than import it.** `rank_bm25` is one pip install away.
But BM25 is about forty lines, every one of its parameters changes the answer, and
this project's rule is that a number which changes the answer has to be visible
and defensible (see `config.py`). An imported black box would make the baseline
un-inspectable, and the baseline is the thing the headline claim is measured
against. It also adds no new dependency: the sparse count matrix comes from
scikit-learn, which is already pinned.

**The formula.** For a query Q and document D:

    score(D, Q) = SUM over terms t in Q of
                      IDF(t) * ( f(t,D) * (k1 + 1) )
                             / ( f(t,D) + k1 * (1 - b + b * |D| / avgdl) )

Three ideas, each doing a specific job:

  * **IDF(t)** — a term appearing in every document tells you nothing. "the" is
    worthless; "adstock" is decisive. IDF weights rare terms up.
  * **k1 (saturation)** — the tenth mention of a word does not make a document ten
    times as relevant. The ratio saturates, so term frequency has diminishing
    returns rather than unbounded ones.
  * **b (length normalisation)** — a long document contains more words and would
    otherwise win by sheer size. b scales the penalty by how far the document's
    length strays from the collection average.

Both k1 and b are set to the standard Lucene/TREC defaults in `config.py`. They
are deliberately *not* tuned: a baseline tuned by the person hoping to beat it is
not a baseline.
"""

from __future__ import annotations

import numpy as np
import scipy.sparse as sp

from src import config


class BM25:
    """A BM25 index over a fixed collection of documents.

    Built once from the same chunk texts the FAISS index was built from, so the
    lexical and dense arms see exactly the same collection in exactly the same
    row order. That row alignment is what lets `retrievers.py` fuse their two
    ranked lists without a join.
    """

    def __init__(self, documents: list[str]):
        from sklearn.feature_extraction.text import CountVectorizer

        # Lowercase, split on word characters, keep every term. No stop-word list
        # and no stemming: BM25's IDF already discounts ubiquitous words, and
        # hand-curated stop lists are a tuning knob this baseline is not allowed.
        self._vectorizer = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b\w\w+\b")

        # tf[d, t] = how many times term t occurs in document d.
        tf = self._vectorizer.fit_transform(documents).astype(np.float32)
        self._n_docs = tf.shape[0]

        doc_lengths = np.asarray(tf.sum(axis=1)).ravel()
        avg_doc_length = float(doc_lengths.mean()) if self._n_docs else 0.0

        # Document frequency: in how many documents does each term appear at all?
        # `.getnnz(axis=0)` counts non-zero entries per column, which is exactly
        # that, and avoids materialising a dense boolean matrix.
        doc_freq = np.asarray((tf > 0).getnnz(axis=0)).ravel().astype(np.float32)

        # Robertson/Sparck-Jones IDF with the +0.5 smoothing terms and a +1 inside
        # the log. The +1 is what keeps the value non-negative: without it, a term
        # occurring in more than half the documents scores negatively and can
        # actively push a matching document *down*, which is not a behaviour
        # anyone wants from a baseline.
        self._idf = np.log(1.0 + (self._n_docs - doc_freq + 0.5) / (doc_freq + 0.5))

        # Precompute the whole denominator-dependent part of the score. Every
        # element of the sparse tf matrix is transformed once, here, instead of
        # once per query — which is what makes query time a sparse matrix-vector
        # product rather than a Python loop over documents.
        tf = tf.tocsr()
        length_norm = 1.0 - config.BM25_B + config.BM25_B * (
            doc_lengths / avg_doc_length if avg_doc_length else np.zeros_like(doc_lengths)
        )
        # Expand each row's normaliser across that row's stored values.
        row_norm = np.repeat(length_norm, np.diff(tf.indptr))
        numerator = tf.data * (config.BM25_K1 + 1.0)
        denominator = tf.data + config.BM25_K1 * row_norm
        weighted = sp.csr_matrix(
            (numerator / denominator, tf.indices, tf.indptr), shape=tf.shape
        )

        # Fold IDF in as a column scaling, so a query becomes a plain sum over the
        # matching columns. `weights[d, t]` is now the full contribution of term t
        # to document d's score.
        self._weights = weighted.multiply(self._idf).tocsc().astype(np.float32)

    @property
    def size(self) -> int:
        """Number of indexed documents."""
        return self._n_docs

    def scores(self, query: str) -> np.ndarray:
        """BM25 score of every document for `query`, as a dense (n_docs,) array.

        A query term the collection has never seen simply has no column and
        contributes nothing — no crash, no special case. A query where *no* term
        is known scores every document 0.0, which the caller sees as a tie.
        """
        # Reuse the fitted vocabulary: `transform`, not `fit_transform`.
        q = self._vectorizer.transform([query])
        term_ids = q.indices
        if term_ids.size == 0:
            return np.zeros(self._n_docs, dtype=np.float32)

        # Sum the precomputed per-term contribution columns. Repeated query terms
        # count once: BM25 scores document term frequency, not query term
        # frequency, for a query this short.
        cols = self._weights[:, np.unique(term_ids)]
        return np.asarray(cols.sum(axis=1)).ravel()

    def top_k(self, query: str, k: int) -> list[tuple[int, float]]:
        """The `k` best-scoring document rows for `query`, best first.

        Uses `argpartition` to find the top k without sorting all ~4,300 scores,
        then sorts only those k. Documents scoring exactly 0 share no terms with
        the query at all and are dropped rather than returned as filler — an
        empty result is a more honest answer than ten arbitrary non-matches.
        """
        scores = self.scores(query)
        k = min(k, scores.size)
        if k == 0:
            return []
        top = np.argpartition(-scores, k - 1)[:k]
        top = top[np.argsort(-scores[top], kind="stable")]
        return [(int(i), float(scores[i])) for i in top if scores[i] > 0.0]
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.lexical'] = _m
setattr(_src, 'lexical', _m)
exec(compile(_code_lexical, 'src/lexical.py', 'exec'), _m.__dict__)

# ---- src/rerank.py --------------------------------------------------
_m = types.ModuleType('src.rerank')
_m.__dict__['__name__'] = 'src.rerank'
_m.__dict__['__file__'] = _ROOT + '/src/rerank.py'
_m.__dict__['__package__'] = 'src'
_code_rerank = r'''"""The cross-encoder reranker — accuracy bought with latency.

Everything else in this project is a **bi-encoder** design: the document is
embedded once at index time, the query is embedded at query time, and relevance
is the similarity between two vectors that never met. That is what makes it fast
— all the document work happens before the query arrives — and it is also its
ceiling. The document's vector was computed without knowing the query, so it has
to be a summary of everything the document might ever be asked about.

A **cross-encoder** does the opposite. It takes the query and the document
*together* as one input, runs them through a transformer that can attend across
both, and outputs a single relevance score. Nothing can be precomputed: every
(query, document) pair is a full forward pass at query time.

    bi-encoder    : encode(query) . encode(document)      <- one vector each
    cross-encoder : model(query + document) -> score      <- one pass per pair

The cross-encoder ranks better, often by a lot, because it can see interactions a
pair of independent summaries cannot. And it is orders of magnitude slower,
because the cost is per candidate rather than per query.

That is why real systems use both, in the arrangement this module implements:
a cheap first stage retrieves a shortlist, and the expensive model only rescores
that shortlist. The size of the shortlist (`config.RERANK_CANDIDATES`) is the
dial between the two. Measuring what that dial buys and what it costs is the
point of the benchmark.

The model is loaded lazily and cached, exactly like `embeddings.get_model` — the
~90 MB download happens on first use, not on import.
"""

from __future__ import annotations

import functools

import numpy as np

from src import config


@functools.lru_cache(maxsize=1)
def get_cross_encoder():
    """Load (and cache) the cross-encoder.

    `ms-marco-MiniLM-L-6-v2` is trained on MS MARCO passage ranking — real search
    queries against real passages — which is the task shape this benchmark
    measures. Import is inside the function so importing this module stays cheap
    and dependency-free, matching the pattern in `embeddings.py`.
    """
    from sentence_transformers import CrossEncoder

    return CrossEncoder(config.RERANK_MODEL)


def rerank_scores(query: str, documents: list[str]) -> np.ndarray:
    """Score every (query, document) pair. Higher is more relevant.

    Returned scores are raw logits, not probabilities and not bounded to any
    range. That is fine and deliberate — they are only ever used to *order* the
    candidates, never compared against another arm's scores or thresholded.
    """
    if not documents:
        return np.zeros(0, dtype=np.float32)
    model = get_cross_encoder()
    scores = model.predict(
        [(query, d) for d in documents],
        show_progress_bar=False,
    )
    return np.asarray(scores, dtype=np.float32)
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.rerank'] = _m
setattr(_src, 'rerank', _m)
exec(compile(_code_rerank, 'src/rerank.py', 'exec'), _m.__dict__)

# ---- src/retrievers.py ----------------------------------------------
_m = types.ModuleType('src.retrievers')
_m.__dict__['__name__'] = 'src.retrievers'
_m.__dict__['__file__'] = _ROOT + '/src/retrievers.py'
_m.__dict__['__package__'] = 'src'
_code_retrievers = r'''"""The four retrieval arms, behind one interface.

The benchmark compares four ways of answering "which chunks are relevant to this
query?". Each is a `Retriever` with a `name` and a `search(query, k)` returning
the same `Retrieved` objects `VectorStore.search` already returns — so the metric
code, the figures, and the notebook never learn which arm they are looking at.

    dense    -> the current system: embed the query, cosine-search FAISS
    bm25     -> lexical term overlap, no neural model at all
    hybrid   -> dense and bm25 fused by reciprocal-rank fusion
    rerank   -> hybrid's shortlist, rescored by a cross-encoder

**The single-definition invariant.** `vectorstore.py` says it is "the single
definition of search the corpus", and `rag.py` and `agent.py` both depend on
that. Nothing here breaks it: `DenseRetriever` *delegates* to
`VectorStore.search` rather than reimplementing cosine search. If it
reimplemented it, the benchmark would be measuring a retriever the shipped system
does not use, and every published number would be about the wrong thing.
`tests/test_retrievers.py` pins this by asserting the two return identical hits.

**Why the arms are ordered this way.** Each one is the previous one plus exactly
one idea, so the benchmark can attribute a change to a cause:

  1. dense alone     — does semantic matching work here?
  2. bm25 alone      — could plain word overlap have done it?
  3. hybrid          — are their mistakes different, or the same mistakes twice?
  4. rerank          — does reading query and document jointly beat comparing
                       two independent summaries, and what does it cost?

An arm that adds two ideas at once is unattributable, which is why there is no
"hybrid + query expansion + rerank" arm.
"""

from __future__ import annotations

from typing import Protocol

import numpy as np
import pandas as pd

from src import config
from src.lexical import BM25
from src.vectorstore import Retrieved, VectorStore


class Retriever(Protocol):
    """Anything that can rank chunks for a query.

    Deliberately the same shape as `VectorStore.search`, so the store itself
    already satisfies it and no adapter is needed for the shipped system.
    """

    name: str

    def search(self, query: str, k: int = config.TOP_K) -> list[Retrieved]:
        ...


def _row_to_retrieved(meta: pd.DataFrame, row: int, score: float) -> Retrieved:
    """Build a `Retrieved` from a metadata row index and a score.

    Every arm ends up here, which is what guarantees they all return the same
    citation fields no matter how they ranked.
    """
    m = meta.iloc[int(row)]
    return Retrieved(
        chunk_id=m["chunk_id"],
        arxiv_id=m["arxiv_id"],
        title=m["title"],
        text=m["text"],
        score=float(score),
    )


# --------------------------------------------------------------------------- #
# 1. Dense — today's system
# --------------------------------------------------------------------------- #
class DenseRetriever:
    """The shipped retriever: embed the query, exact cosine search over FAISS.

    A thin delegation on purpose. This arm must be *the same code path* the RAG
    pipeline and the agent use, or the benchmark is measuring a lookalike.
    """

    name = "dense"

    def __init__(self, store: VectorStore):
        self._store = store

    def search(self, query: str, k: int = config.TOP_K) -> list[Retrieved]:
        return self._store.search(query, k=k)


# --------------------------------------------------------------------------- #
# 2. BM25 — the lexical baseline
# --------------------------------------------------------------------------- #
class BM25Retriever:
    """Rank by term overlap. No embeddings, no model, no GPU.

    Built from the vector store's own metadata frame so the two arms index the
    same chunks in the same row order — which is what lets the hybrid arm fuse
    them by row id.
    """

    name = "bm25"

    def __init__(self, store: VectorStore):
        self._meta = store.meta
        self._bm25 = BM25(self._meta["text"].tolist())

    def search(self, query: str, k: int = config.TOP_K) -> list[Retrieved]:
        return [
            _row_to_retrieved(self._meta, row, score)
            for row, score in self._bm25.top_k(query, k)
        ]


# --------------------------------------------------------------------------- #
# 3. Hybrid — reciprocal-rank fusion
# --------------------------------------------------------------------------- #
class HybridRRFRetriever:
    """Fuse the dense and lexical rankings by reciprocal-rank fusion.

    **Why fuse ranks and not scores.** A cosine similarity lives in [-1, 1]; a
    BM25 score is unbounded and depends on the collection's term statistics. They
    are not on a common scale and no fixed weighting makes them comparable, so
    adding or averaging them is meaningless. RRF throws the scores away and keeps
    only the *positions*:

        score(d) = SUM over arms of  1 / (RRF_K + rank of d in that arm)

    A document ranked 1st by one arm and unranked by the other still scores well;
    a document ranked 3rd by *both* can beat it. That is the behaviour worth
    having — agreement between two different kinds of evidence is a stronger
    signal than one arm's confidence.

    `RRF_K = 60` (Cormack et al., 2009) damps the top ranks so a single arm's
    confident-but-wrong first hit cannot dominate the fusion.

    Each arm is asked for a *deeper* list than the caller wants, because a
    document must be ranked by at least one arm to be fusable at all — fusing two
    top-10 lists can only ever consider 20 candidates.
    """

    name = "hybrid"

    def __init__(self, store: VectorStore, depth: int = config.RERANK_CANDIDATES):
        self._store = store
        self._meta = store.meta
        self._bm25 = BM25(self._meta["text"].tolist())
        self._depth = depth

    def _fused_rows(self, query: str, depth: int) -> list[tuple[int, float]]:
        """Fused (row, rrf_score) pairs, best first — the shared ranking step."""
        dense_rows = self._store.search_rows(query, k=depth)
        bm25_rows = self._bm25.top_k(query, depth)

        fused: dict[int, float] = {}
        for ranking in (dense_rows, bm25_rows):
            for rank, (row, _score) in enumerate(ranking, start=1):
                fused[row] = fused.get(row, 0.0) + 1.0 / (config.RRF_K + rank)

        return sorted(fused.items(), key=lambda kv: -kv[1])

    def search(self, query: str, k: int = config.TOP_K) -> list[Retrieved]:
        ranked = self._fused_rows(query, self._depth)[:k]
        return [_row_to_retrieved(self._meta, row, score) for row, score in ranked]


# --------------------------------------------------------------------------- #
# 4. Rerank — cross-encoder over the hybrid shortlist
# --------------------------------------------------------------------------- #
class RerankRetriever:
    """Retrieve a shortlist cheaply, then rescore it with a cross-encoder.

    The standard production arrangement, and the only arm here whose cost scales
    with the shortlist rather than the collection. It reranks the *hybrid* arm's
    candidates rather than the dense arm's, so recall going into the reranker is
    as high as this project can make it — a cross-encoder can only reorder what it
    is given, and cannot rescue a document the first stage never retrieved.
    """

    name = "rerank"

    def __init__(self, hybrid: HybridRRFRetriever,
                 candidates: int = config.RERANK_CANDIDATES):
        self._hybrid = hybrid
        self._meta = hybrid._meta
        self._candidates = candidates

    def search(self, query: str, k: int = config.TOP_K) -> list[Retrieved]:
        from src.rerank import rerank_scores

        shortlist = self._hybrid._fused_rows(query, self._candidates)[:self._candidates]
        if not shortlist:
            return []

        rows = [row for row, _ in shortlist]
        texts = [self._meta.iloc[r]["text"] for r in rows]
        scores = rerank_scores(query, texts)

        order = np.argsort(-scores, kind="stable")[:k]
        return [_row_to_retrieved(self._meta, rows[i], float(scores[i])) for i in order]


# --------------------------------------------------------------------------- #
# Assembly
# --------------------------------------------------------------------------- #
def build_all(store: VectorStore) -> list[Retriever]:
    """Construct all four arms, sharing what can be shared.

    The BM25 index is built twice (once for the lexical arm, once inside the
    hybrid arm) — a couple of seconds, and worth it to keep each arm a
    self-contained object that can be constructed and reasoned about alone.
    Returned in increasing order of cost, which is the order the figures use.
    """
    hybrid = HybridRRFRetriever(store)
    return [
        DenseRetriever(store),
        BM25Retriever(store),
        hybrid,
        RerankRetriever(hybrid),
    ]
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.retrievers'] = _m
setattr(_src, 'retrievers', _m)
exec(compile(_code_retrievers, 'src/retrievers.py', 'exec'), _m.__dict__)

# ---- src/significance.py --------------------------------------------
_m = types.ModuleType('src.significance')
_m.__dict__['__name__'] = 'src.significance'
_m.__dict__['__file__'] = _ROOT + '/src/significance.py'
_m.__dict__['__package__'] = 'src'
_code_significance = r'''"""Is the gap between two arms real, or is it the question set?

`evaluation.summarise_arms` gives every arm a bootstrap interval of its own, and
the README then prints four of them in a table. That invites a comparison the
intervals cannot support. **Two marginal intervals overlapping does not mean the
difference is not significant** — it is a well-known trap, and here it would be a
self-inflicted one, because the arms all answered the *identical* question list.
When the same questions go to every arm, the per-question *difference* is
available, and it is a far tighter measurement than either arm's spread:

    arm A on q17: 0.63     arm B on q17: 0.50     difference: +0.13
    arm A on q18: 0.99     arm B on q18: 0.86     difference: +0.13

Both arms vary hugely across questions (0.63 vs 0.99 — some questions are simply
harder), and that variance is what widens the marginal intervals. It cancels
exactly in the difference. Pairing is not a refinement here; it is the difference
between a test that can see the effect and one that cannot.

## What this module reports, per pair of arms

1. **Mean difference in nDCG@10**, with a percentile bootstrap interval over the
   *differences* — the effect size, in the metric's own units. This is the number
   worth quoting: "BM25 beats dense by 0.23 [0.17, 0.28] nDCG".
2. **A two-sided paired randomization test.** Under the null "the two arms are
   interchangeable", the sign of each question's difference is arbitrary, so
   flipping signs at random generates the null distribution directly. It is the
   standard significance test in IR (Smucker et al., 2007, found it and the
   bootstrap agree closely and both beat the t-test's assumptions), and it needs
   nothing from the data but exchangeability — no normality, no equal variance.
3. **A Holm-corrected p-value.** Four arms is six pairs; five would be ten. Ten
   independent tests at 0.05 give a ~40% chance of at least one false positive,
   so an uncorrected table of pairwise wins is a table of some wins and some
   noise. Holm is used rather than Bonferroni because it is uniformly more
   powerful at the same guarantee, and the family is stated explicitly: all
   pairwise comparisons within one run, on one metric.

## What it deliberately does not do

No test is run *across* the two question sets. The generated known-item queries
and the 11 hand-written ones are different tasks with different label provenance
(`eval/ADJUDICATION.md`), and pooling them to grow n would buy significance by
mixing populations — the exact move this module exists to make unnecessary.
"""

from __future__ import annotations

from itertools import combinations

import numpy as np
import pandas as pd

from src import config


#: Sign-flip resamples for the randomization test. With 180 questions the exact
#: test would need 2**180 flips, so it is sampled; 10,000 gives a p-value stable
#: to about ±0.004 at p = 0.05, which is finer than any decision made from it.
N_PERMUTATIONS = 10_000


def paired_differences(rows: list[dict], arm_a: str, arm_b: str,
                       metric: str = "ndcg@k") -> tuple[np.ndarray, list[str]]:
    """Per-question `metric` differences (arm_a − arm_b), aligned by question id.

    Returns the difference vector and the qids it came from, in a fixed order.
    Alignment is by qid rather than by position: the benchmark writes rows arm by
    arm and nothing guarantees the two blocks are in the same order, and a paired
    test on mis-paired rows is worse than no test — it looks fine and means
    nothing.
    """
    df = pd.DataFrame(rows)
    a = df[df.arm == arm_a].set_index("qid")[metric]
    b = df[df.arm == arm_b].set_index("qid")[metric]

    common = sorted(set(a.index) & set(b.index))
    if len(common) != len(a) or len(common) != len(b):
        raise ValueError(
            f"{arm_a} and {arm_b} were not scored on the same questions "
            f"({len(a)} vs {len(b)}, {len(common)} shared) — the pairing that "
            "makes this test valid does not hold."
        )
    diffs = (a.loc[common] - b.loc[common]).to_numpy(dtype=float)
    return diffs, common


def randomization_p_value(diffs: np.ndarray,
                          n_permutations: int = N_PERMUTATIONS,
                          seed: int = config.SEED) -> float:
    """Two-sided paired randomization (sign-flip) test p-value.

    The null is that the two arms are interchangeable, so for any question the
    observed difference was as likely to have come out with the opposite sign.
    Flipping signs uniformly at random therefore samples the null distribution of
    the mean difference, with no distributional assumption at all.

    The +1 in both numerator and denominator is the standard correction: the
    observed arrangement is itself one of the possible sign assignments, and
    excluding it can return p = 0, which is a claim no finite resample supports.
    """
    diffs = np.asarray(diffs, dtype=float)
    if diffs.size == 0:
        return 1.0
    observed = abs(float(diffs.mean()))

    rng = np.random.default_rng(seed)
    # One (n_permutations, n) matrix of ±1, then a row-wise mean: the whole test
    # is two array operations.
    signs = rng.choice([-1.0, 1.0], size=(n_permutations, diffs.size))
    null_means = np.abs((signs * diffs).mean(axis=1))
    n_at_least = int((null_means >= observed - 1e-12).sum())
    return (n_at_least + 1) / (n_permutations + 1)


def holm_adjust(p_values: list[float]) -> list[float]:
    """Holm-Bonferroni step-down adjustment, returned in the input order.

    Sort ascending, multiply the i-th smallest by (m − i), then enforce
    monotonicity so an adjusted p-value can never fall below one that preceded
    it. Controls the family-wise error rate at the same level as Bonferroni while
    rejecting at least as much — there is no reason to prefer Bonferroni here.
    """
    m = len(p_values)
    if m == 0:
        return []
    order = sorted(range(m), key=lambda i: p_values[i])
    adjusted = [0.0] * m
    running = 0.0
    for rank, idx in enumerate(order):
        value = min(1.0, (m - rank) * p_values[idx])
        running = max(running, value)   # step-down monotonicity
        adjusted[idx] = running
    return adjusted


def compare_arms(rows: list[dict], metric: str = "ndcg@k",
                 alpha: float = 0.05) -> list[dict]:
    """Every pairwise arm comparison, with effect size, interval and p-value.

    One record per unordered pair, ordered so `mean_diff` is positive and `arm_a`
    is the winner — a table where every row reads "A beat B by X" needs no sign
    convention explained to it.
    """
    from src.evaluation import bootstrap_ci

    df = pd.DataFrame(rows)
    arms = [a for a in df["arm"].unique()]

    records: list[dict] = []
    for arm_a, arm_b in combinations(arms, 2):
        diffs, qids = paired_differences(rows, arm_a, arm_b, metric)
        if diffs.mean() < 0:                      # orient so arm_a wins
            arm_a, arm_b, diffs = arm_b, arm_a, -diffs

        lo, hi = bootstrap_ci(diffs.tolist())
        records.append({
            "arm_a": arm_a,
            "arm_b": arm_b,
            "metric": metric,
            "n_questions": int(diffs.size),
            "mean_diff": float(diffs.mean()),
            "mean_diff_lo": lo,
            "mean_diff_hi": hi,
            "n_wins_a": int((diffs > 0).sum()),
            "n_wins_b": int((diffs < 0).sum()),
            "n_ties": int((diffs == 0).sum()),
            "p_value": randomization_p_value(diffs),
        })

    adjusted = holm_adjust([r["p_value"] for r in records])
    for record, p_adj in zip(records, adjusted):
        record["p_value_holm"] = p_adj
        record["significant"] = bool(p_adj < alpha)
        # An interval on the difference that excludes zero and a significant test
        # are two routes to the same conclusion. They are reported together
        # because when they disagree - which happens near the boundary - that
        # disagreement is information, not something to resolve by picking one.
        record["ci_excludes_zero"] = bool(record["mean_diff_lo"] > 0)

    records.sort(key=lambda r: -r["mean_diff"])
    return records


def significance_table(records: list[dict]) -> pd.DataFrame:
    """The comparison records as a printable frame, for the notebook.

    Column labels are ASCII on purpose. A Windows console defaults to cp1252 and
    raises on a delta or a minus sign, so a table that renders in Jupyter would
    crash the same code at a terminal — and this project's numbers are meant to
    be checkable from either.
    """
    return pd.DataFrame([
        {
            "comparison": f"{r['arm_a']} vs {r['arm_b']}",
            "diff nDCG@10": round(r["mean_diff"], 3),
            "95% CI": f"[{r['mean_diff_lo']:.3f}, {r['mean_diff_hi']:.3f}]",
            "wins/losses/ties": f"{r['n_wins_a']}/{r['n_wins_b']}/{r['n_ties']}",
            "p (Holm)": f"{r['p_value_holm']:.4f}",
            "significant": "yes" if r["significant"] else "no",
        }
        for r in records
    ])
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.significance'] = _m
setattr(_src, 'significance', _m)
exec(compile(_code_significance, 'src/significance.py', 'exec'), _m.__dict__)

print('inlined modules:', ['config', 'acquisition', 'llm', 'embeddings', 'vectorstore', 'rag', 'agent', 'chunking', 'evalset', 'evaluation', 'lexical', 'rerank', 'retrievers', 'significance'])

# Agentic RAG Research Assistant — capstone

**A question-answering system over 2,012 arXiv machine-learning papers that
retrieves evidence before it answers, cites its sources, and — agentically —
decides for itself how to search.**

> **Executive summary — three findings.**
>
> **1. The retriever this system ships is the wrong default.** Benchmarked against
> three alternatives on 191 questions, a 1990s lexical baseline (BM25) beats it on
> every metric at one-eighth the latency: **nDCG@10 0.978 [0.962, 0.992] at 2.0 ms**
> versus **0.751 [0.691, 0.811] at 17.4 ms**. Paired per question, that gap is
> +0.227 with **52 wins to 1 loss**, p < 0.001. The cross-encoder reranker — which
> an earlier version of this project recommended as the obvious next step — buys
> **no measurable quality over BM25** (+0.014, p = 0.10) while charging about
> **1,200× its latency**.
>
> **2. A single retrieval structurally fails on a compound question.** Single-shot
> retrieval recovers only **60%** of the needed papers on multi-hop questions,
> because one search embeds the whole compound question as a single point.
> Swapping the retriever recovers part of that gap; the rest is what the **agent**
> is for.
>
> **3. The project's own drift monitor was broken, and acting on it is what proved
> it.** The probe read a 27.2% similarity drop against a 15% re-index trigger, so
> the index was rebuilt — and the same probe read 26.7% on an index one hour old.
> The metric had a structural floor and could never have said anything else. It now
> triggers on excess over a *measured* floor.

This notebook runs top-to-bottom on a fresh kernel. It imports everything from
`src/` (so it can never disagree with the scripts and tests), reproduces the
retrieval numbers **live and locally**, and treats the Claude-backed answer steps
as reference-only — they run if `ANTHROPIC_API_KEY` is set and otherwise degrade
gracefully, exactly like the portfolio's GPU-trained notebooks.

This is a **hybrid** project: the RAG pipeline is the engineering track, and the
**evaluation harness carries it** — the questions it answers are which retrieval
strategy is actually worth its latency, whether the gaps between them survive a
significance test, where the cutoffs should sit, whether the chunking constant was
worth anything, and whether a multi-step agent beats a single-shot one.


## 0 · Setup

Make `src/` importable and load the config — the single place every tuned constant lives.

In [ ]:
import sys, json
from pathlib import Path

# Make the project root importable whether the kernel started in notebooks/ or root.
ROOT = Path.cwd()
if (ROOT / "src").exists():
    pass
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config
print("Project root:", ROOT)
print("Embedding model:", config.EMBED_MODEL, f"(dim {config.EMBED_DIM})")
print("Chunk size / overlap:", config.CHUNK_SIZE_CHARS, "/", config.CHUNK_OVERLAP_CHARS, "chars")
print("Retrieval top-k:", config.TOP_K, " | max agent steps:", config.MAX_AGENT_STEPS)
print("Generation model:", config.GEN_MODEL, " | API key present:", config.api_key_available())

## 1 · Problem framing

A large language model answering from memory is a student sitting a closed-book
exam: fluent, and confidently wrong on specifics. **RAG turns it open-book** — fetch
the relevant passages first, then answer *from them*, with citations.

The decision this project interrogates is one level up: **once you retrieve, should
you retrieve *once* (naive RAG) or let the model decide how to retrieve, iterating
as needed (agentic RAG)?** The agent is more capable but costs more calls. Is it
worth it — and *where*? Our sub-questions:

1. Where does single-shot retrieval actually fail? *(§4 — the finding)*
2. Can an agent that decomposes and re-searches recover those failures? *(§5)*
3. What does that cost? *(§5 — the trade-off)*

## 2 · Data — the corpus and its provenance

We index recent cs.CL / cs.LG papers from the arXiv API, plus 12 seeded seminal
papers (Transformer, BERT, RAG, DPR, ReAct, …) so the evaluation's ground-truth
answers are guaranteed present. Everything is loaded through `src/` with the same
provenance the pipeline records — nothing is re-implemented here.

In [ ]:
from src import acquisition

prov = json.load(open(config.PROVENANCE_FILE))
print("Source:", prov["source"])
print("Categories:", prov["categories"])
print("Downloaded (UTC):", prov["downloaded_utc"])
print("Papers:", prov["paper_count"])
print("Corpus SHA-256:", prov["papers_sha256"][:16], "...")

papers = acquisition.load_papers()
print("\nExample seeded paper:")
seed = next(p for p in papers if p.arxiv_id == "2005.11401")  # the RAG paper
print(" ", seed.arxiv_id, "—", seed.title)
print(" ", seed.abstract[:220], "...")

## 3 · The RAG pipeline — how a question finds its evidence

Build-time (already run by `run_pipeline.py`): **chunk → embed → index**. We reuse
`inspect.getsource` to show the real chunking code, so the explanation can never
drift from what actually ran.

In [ ]:
import inspect
from src import chunking

print(inspect.getsource(chunking.split_text))

Every chunk carries its paper's id and title — that metadata is what lets an
answer *cite* the passage it used. Now load the built index (the vector store) and
run one live retrieval to see semantic search work.

In [ ]:
from src.vectorstore import VectorStore

store = VectorStore.load()
print("Index:", store.size, "chunks\n")

# A query whose words barely overlap the target passage — semantic search, not keywords.
hits = store.search("how do neural networks pay attention to parts of a sequence", k=3)
for h in hits:
    print(f"[{h.arxiv_id}] score={h.score:.3f}  {h.title}")
    print("   ", h.text[:150], "...\n")

## 4 · EDA / the finding — where single-shot retrieval breaks

This is the load-bearing result. For each evaluation question we know which
paper(s) genuinely answer it (`eval/qa_set.json`), so we can measure whether
retrieval surfaced them — **hit@k**, **recall@k**, **MRR** — all computed locally
with no API key, through `src/evaluation.py`.

In [ ]:
from src import evaluation as ev
import pandas as pd

questions = ev.load_eval_set()
retrieval_rows = ev.evaluate_retrieval(store, questions)
df = pd.DataFrame(retrieval_rows)

summary = df.groupby("hop")[["hit@k", "recall@k", "mrr"]].mean().round(3)
summary.loc["all"] = df[["hit@k", "recall@k", "mrr"]].mean().round(3)
summary

**Read this table carefully — it is the whole argument.** Multi-hop recall@5 is
**0.60**: when a question needs two papers, a single search typically grabs only
one of them. The compound question embeds to one point in the vector space; the
second, differently-phrased piece of evidence lives elsewhere and never surfaces.

Single-hop recall is **0.83**, and the single question that fails is worth more
than the average. It is `s5` — the *word2vec* question — and section 5 will
identify it independently, for a completely different reason, as the corpus's one
vocabulary-mismatch case. Two measurements taken for different purposes landing on
the same row is the strongest kind of corroboration a small eval set can offer.

The per-question breakdown makes the failure concrete — the rows with recall below
1.0 are the ones to read:


In [ ]:
df[df.hop == "multi"][["qid", "hit@k", "recall@k", "mrr"]].round(2)

On `m4` (attention *before* vs *after* the Transformer) single-shot retrieval
recall is **0.0** — it missed both papers. That is precisely the kind of question an
agent should rescue by searching for "attention before the Transformer" separately
from "what the Transformer changed". The committed figure visualizes the gap:


In [ ]:
from IPython.display import Image
Image(filename=str(config.FIGURES_DIR / "01_retrieval_metrics.png"))

## 5 · Benchmarking retrieval — four arms, and the baseline that won

Section 4 showed *the shipped retriever* failing on multi-hop questions. It never
asked the more basic question: **is this retriever any good in the first place?**

Almost no public RAG notebook asks it either. The pattern is to wire an embedding
model to a vector store, retrieve top-k, hand it to an LLM and show one impressive
answer — with no baseline to beat, no confidence interval, and no latency. A RAG
demo is a commodity; a defensible evaluation of one is not.

So we benchmark four arms, each the previous one **plus exactly one idea**, so a
change in the score can be attributed to a cause:

| Arm | What it adds | Implemented in |
|---|---|---|
| `dense` | semantic matching | delegates to `VectorStore.search` — the *same* code path the system ships |
| `bm25` | literal term overlap | `src/lexical.py` — BM25 Okapi written out, k1/b at the Lucene defaults |
| `hybrid` | fusing the two rankings | reciprocal-rank fusion, `Σ 1/(60 + rank)` |
| `rerank` | reading query and document jointly | cross-encoder over the hybrid shortlist |

An arm that added two ideas at once would be unattributable — which is why there
is no "hybrid + query expansion + reranking" arm.

In [ ]:
from src import retrievers

arms = retrievers.build_all(store)
demo_q = "What neural network architecture did Attention Is All You Need introduce?"

for arm in arms:
    hits = arm.search(demo_q, k=3)
    print(f"[{arm.name}]")
    for h in hits:
        print(f"   {h.arxiv_id}  {h.score:>8.3f}  {h.title[:64]}")
    print()

**Already visible on a single query:** the dense arm — the one this project ships —
does not return *Attention Is All You Need* for a question that names it. BM25 puts
it first. That is not a fluke, and the rest of this section quantifies it.

### The full benchmark

Scored on **180 generated known-item queries** (`src/evalset.py`) plus the 11
hand-written ones, at k=10, with 2,000-resample bootstrap intervals.

The cross-encoder arm takes ~2.6 s per query, so the full run is a script rather
than a notebook cell — the same "reference-only, load the saved result" convention
this portfolio uses for GPU-trained notebooks. Regenerate with
`python run_retrieval_bench.py`; every number below is read from its committed
output.

In [ ]:
import json

with open(config.ARM_SUMMARY_FILE, encoding="utf-8") as f:
    arm_summary = pd.DataFrame(json.load(f))

overall = arm_summary[arm_summary.slice == "all"].set_index("arm")
overall["nDCG@10 (95% CI)"] = [
    f"{r['ndcg@k']:.3f}  [{r['ndcg@k_lo']:.3f}, {r['ndcg@k_hi']:.3f}]"
    for _, r in overall.iterrows()
]
overall[["nDCG@10 (95% CI)", "ap@k", "recall@k", "mrr", "latency_ms_median"]].rename(
    columns={"ap@k": "MAP", "recall@k": "recall@10",
             "mrr": "MRR", "latency_ms_median": "median ms"}
).round(3)

**The 1990s lexical baseline beats the embedding model this system ships — on
every metric, at one-eighth the latency.**

Resist reading the intervals against each other, though. Two marginal intervals
overlapping does *not* mean two arms are indistinguishable, and the next cells
make that comparison properly. First, the trade-off itself:


In [ ]:
Image(filename=str(config.FIGURES_DIR / "05_quality_vs_latency.png"))

In [ ]:
from src import significance

with open(config.SIGNIFICANCE_FILE, encoding="utf-8") as f:
    sig_records = json.load(f)

significance.significance_table(sig_records)

**The bottom row changed a conclusion.** On the point estimates BM25 (0.978) sits
above the cross-encoder (0.964), and an earlier version of this project said so —
that the reranker "scores lower". Paired, that gap is **8 wins against 6 losses
over 166 ties**, and it does not survive its own significance test.

So the defensible claim is not that BM25 beats the reranker. It is:

> **the cross-encoder buys no measurable quality over BM25, and charges about
> 1,200× the latency for it.**

That is the *stronger* statement. "Slightly better" invites an argument about
whether the difference is worth paying for; "no measurable difference at a
thousand times the price" ends it.

Note also the wins/losses column, which a mean cannot show you: `hybrid vs dense`
is **46–0**. An arm that never once loses is a different thing from an arm that
wins twice enormously, and only the per-question view distinguishes them.

In [ ]:
Image(filename=str(config.FIGURES_DIR / "08_arm_significance.png"))

BM25 is the top-left point: **nothing else buys quality it does not already have,
and everything else costs more.**

### Is any of that gap real?

Every arm answered the *same* 180 questions, so the per-question **difference**
exists — and it is a much sharper instrument than four intervals side by side.
Question difficulty is shared by both arms in any pair, so it cancels in the
difference; that shared variance is most of what makes the marginal intervals
wide.

`src/significance.py` compares every pair with a **sign-flip randomization test**
(the null is that the two arms are interchangeable, so the sign of each question's
difference is arbitrary — flip them at random 10,000 times and see how extreme the
real mean is) and a **Holm correction** across all six pairs, because six tests at
α = 0.05 is a ~26% chance of at least one false positive.


In [ ]:
Image(filename=str(config.FIGURES_DIR / "06_query_style_robustness.png"))

In [ ]:
with open(config.K_SWEEP_FILE, encoding="utf-8") as f:
    sweep = pd.DataFrame(json.load(f))

sweep.pivot(index="k", columns="arm", values="recall@k")[
    ["dense", "bm25", "hybrid", "rerank"]
].round(3)

Three answers no fixed-k table could give:

1. **Serving 5 costs nothing under BM25** — its recall is already 1.000 at k=5, so
   the gap between the cutoff the system serves and the one the benchmark reports
   hides no losses. Under the *dense* arm it would (0.778 at 5 against 0.800 at 10).
2. **The dense arm never saturates.** At k=50 it has found 85.6% of the target
   papers. That is a **recall ceiling**, not a ranking problem — and it is the
   mechanical reason a reranker cannot rescue it. A cross-encoder reorders what it
   is handed; it cannot retrieve what the first stage never returned.
3. **The reranker's shortlist is over-provisioned.** Hybrid recall reaches 1.000 by
   k=20, so nothing beyond rank 20 is ever needed. `RERANK_CANDIDATES` could be 20
   instead of 50 — roughly 2.5× less cross-encoder work at no cost in recall.

In [ ]:
Image(filename=str(config.FIGURES_DIR / "09_k_sweep.png"))

BM25 never reads word order at all, so it barely moves (**−0.065**). The dense arm
loses **two-thirds of its quality** (0.969 → 0.291) once the query stops being a
sentence. A bi-encoder trained on natural prose has no particular reason to handle
a keyword bag well, and it doesn't.

### How deep do you have to look? The cutoff sweep

Everything so far is reported at k=10. Why 10? Because BEIR and TREC report there —
a fine reason to *report* at 10 and no reason at all to believe it is where the
interesting behaviour is. Meanwhile the system serves the top **5** chunks to the
LLM (`config.TOP_K`) and the reranker rescores a shortlist of **50**
(`config.RERANK_CANDIDATES`). Three constants, all argued in `config.py`, none
measured against each other.

`evaluation.sweep_k` searches **once** at k=50 per (arm, question) and truncates
the ranking at each cutoff — so the *cutoff* is the only thing varying. Searching
separately per k would confound it with "a retriever configured for k", which the
reranker genuinely is.


In [ ]:
with open(config.RESULTS_DIR / "arm_comparison_curated.json", encoding="utf-8") as f:
    curated_rows = pd.DataFrame(json.load(f))

curated_rows.pivot(index="qid", columns="arm", values="recall@k")[
    ["dense", "bm25", "hybrid", "rerank"]
].join(curated_rows.groupby("qid")["hop"].first()).sort_values(["hop", "qid"])

Two rows matter more than the average.

**`s5` — the one question BM25 cannot answer.** It asks what **word2vec** learns.
The correct paper is *"Efficient Estimation of Word Representations in Vector
Space"*, and the string `word2vec` appears **zero times in all 4,298 chunks**. BM25
cannot match a term that is not there and scores 0.00; the dense arm does find it,
around rank 6, because *word2vec* and *word representations in vector space* are
near-neighbours in embedding space.

That is the canonical **vocabulary-mismatch** case — the exact problem dense
retrieval was invented to solve — appearing live in an evaluation whose headline
says dense retrieval lost. Hybrid RRF inherits BM25's failure rather than dense's
success: fusion is not free insurance.

It is also the same question section 4 found failing at `TOP_K = 5`, where rank 6
is one place outside what the system actually serves. So on this corpus `s5` is
unanswerable by the shipped system *and* by the lexical baseline, for opposite
reasons.

**`m4` — the question section 4 showed failing completely.** Dense scores 0.00; BM25
recovers one of the two papers. Swapping the retriever fixes half of the multi-hop
gap with no agent at all — and the remaining half is a genuine two-paper problem
that no single retrieval strategy solves. That residue is the agent's case.

### Does the chunk size actually matter?

`config.CHUNK_SIZE_CHARS = 1000` carries a paragraph of justification — too large
and one vector must average several ideas, too small and a sentence loses the
context that made it meaningful — and, until this run, zero measurement. It is the
one build-time decision that moves *both* arms, for different reasons: embedding
dilution for dense, and document-length normalisation for BM25, since chunk size
*is* document length.

`run_ablation.py` rebuilds the corpus at five sizes — re-chunk, re-embed, re-index
each time — and scores dense and BM25 on the same questions. The cross-encoder is
excluded on cost, and that exclusion is stated rather than left to be noticed.


In [ ]:
with open(config.CHUNK_ABLATION_FILE, encoding="utf-8") as f:
    ablation = pd.DataFrame(json.load(f))

ablation.pivot(index="chunk_size", columns="arm", values="ndcg@k").round(3).join(
    ablation.groupby("chunk_size")["n_chunks"].first()
)

In [ ]:
Image(filename=str(config.FIGURES_DIR / "10_chunk_ablation.png"))

**The two arms want opposite things**, and the same paired discipline as before
says which of the two movements is real:

In [ ]:
with open(config.ABLATION_SIGNIFICANCE_FILE, encoding="utf-8") as f:
    ablation_tests = pd.DataFrame(json.load(f))

ablation_tests[["arm", "chunk_size", "mean_diff", "mean_diff_lo", "mean_diff_hi",
                "n_better", "n_worse", "n_ties", "p_value_holm", "significant"]].round(4)

### How long do these numbers stay true?

Every number above describes an index built on a particular day, over a literature
that does not stop. `src/drift.py` compares mean top-1 similarity for queries drawn
from **indexed** papers against queries drawn from **newer, unseen** ones, and
turns the gap into a re-indexing trigger.

**This is the part of the project that was wrong, and finding out required acting
on it.** The previous index (2026-07-27) read a 27.2% drop against a flat 15%
trigger, so the README recommended a rebuild. The rebuild happened — against a
corpus sharing not one paper with the old one:


In [ ]:
with open(config.RESULTS_DIR / "corpus_turnover.json", encoding="utf-8") as f:
    turnover = json.load(f)

print(f"old index      : {turnover['old_index_built_utc'][:10]}")
print(f"new index      : {turnover['new_index_built_utc'][:10]}")
print(f"papers sampled : {turnover['n_sampled']} (the old eval set's target papers)")
print(f"still present  : {turnover['n_still_present']}  "
      f"-> {turnover['survival_rate']:.0%} survived")
print(f"seeded papers  : {turnover['seeded_papers_present']}"
      f"/{turnover['seeded_papers_expected']} present in both, by construction")

**Complete turnover in 16 days.** The corpus is the newest `MAX_PAPERS` in cs.CL
and cs.LG, and arXiv publishes more than that in a fortnight, so the window rotated
entirely. Only the 12 papers pinned by id survived — which is exactly why they are
pinned, and why the 11 curated questions still mean the same thing after a rebuild.

So: a brand-new index, over a completely different corpus. Here is what the drift
probe says about it.

In [ ]:
with open(config.DRIFT_RESULTS_FILE, encoding="utf-8") as f:
    drift_record = json.load(f)

print(f"indexed papers : {drift_record['baseline_top1_mean']:.3f} mean top-1 similarity")
print(f"newer papers   : {drift_record['probe_top1_mean']:.3f}")
print(f"relative drop  : {drift_record['relative_drop']:.1%}")
print(f"day-zero floor : {drift_record['floor_relative_drop']:.1%}"
      "   <- what this probe reads on a brand-new index")
print(f"excess         : {drift_record['excess_over_floor']:+.2%}"
      f"  (trigger at {drift_record['trigger_excess']:+.0%})")
print(f"verdict        : {'RE-INDEX' if drift_record['reindex_now'] else 'at the floor - nothing to do'}")

Image(filename=str(config.FIGURES_DIR / "07_drift.png"))

**The drop barely moved: 27.2% on the old index, 26.7% on one built an hour ago.**

The largest freshness intervention that exists — replacing every paper in the
corpus — changed the metric by half a percentage point. That is not an index
result, it is a *metric* result: the raw drop was never measuring staleness.

The reason is structural. A baseline query is a sentence lifted verbatim out of a
document sitting *in* the index, so its top-1 match is a near-self-match by
construction. A probe query comes from a document that is absent by definition, so
its best match is the closest topical neighbour. The gap between "verbatim
self-match" and "nearest neighbour" is a property of the measurement, present in
full on day zero — so a threshold on it fires on every index, always, and
distinguishes nothing.

The trigger now tests the **excess over a measured day-zero floor** (0.267 —
observed on the fresh index, not assumed). This index reads **+0.05 percentage
points**: nothing to do.

Two things worth taking from this beyond the fix:

- **The bug was findable from the start.** `drift.py`'s own docstring always said
  "the gap *widening* is the signal", and the code thresholded the level anyway.
  The prose was right and unread; the code was wrong and trusted.
- **The floor expires.** Change the embedding model, the chunk size or the corpus
  size and it silently stops being the right number, while the constant in
  `config.py` stays put. That makes it the most fragile line in the project, which
  is written down next to it rather than left to be discovered.

> **How the questions were labelled, and where they mislead**, is written up in
> [`eval/ADJUDICATION.md`](../eval/ADJUDICATION.md). The short version: every
> generated query is built from its target document's own words, which
> structurally favours BM25. Three things make the finding survivable — the three
> query styles measure a *slope* rather than a single point, the hand-written set
> (which does not share that bias) agrees, and the whole result replicated on a
> corpus with zero overlap.


## 6 · Method & the head-to-head — naive vs agentic RAG

Both systems share the **same** `VectorStore.search` (one definition of retrieval in
the repo), so any difference between them is about the *strategy*, not the search.

**Naive RAG** retrieves once and generates once. **Agentic RAG** runs a from-scratch
Claude tool-use loop: it decomposes the question, searches for each part, reads the
results, searches again if needed (up to `MAX_AGENT_STEPS`), then answers. Here is
the actual loop that does it — no framework, fully inspectable:

In [ ]:
from src import agent as agent_mod
print(inspect.getsource(agent_mod.answer))

### Running the two systems

The retrieval above is local and always reproduces. **Generation needs an Anthropic
API key.** The next cell is *reference-only*: with a key it runs both systems live
and shows the agent's search trace; without one it explains what would run and
skips gracefully, so Restart & Run All succeeds either way.

In [ ]:
from src import rag
from src.llm import LLMKeyMissing

question = ("How does retrieval-augmented generation combine a dense passage "
            "retriever with a sequence-to-sequence generator?")

if config.api_key_available():
    naive = rag.answer(store, question)
    print("=== NAIVE RAG ===")
    print(naive.answer)
    print("cited:", naive.cited_ids, "\n")

    agentic = agent_mod.answer(store, question)
    print("=== AGENTIC RAG — search trace ===")
    for i, step in enumerate(agentic.steps, 1):
        print(f"  step {i}: {step.query!r} -> {[h.arxiv_id for h in step.hits]}")
    print(f"  (stopped: {agentic.stopped_reason})")
    print("\n=== AGENTIC ANSWER ===")
    print(agentic.answer)
    print("cited:", agentic.cited_ids)
else:
    print("No ANTHROPIC_API_KEY set — generation is reference-only in this run.")
    print("With a key, this cell runs both systems and prints the agent's search")
    print("trace: you would see it search for the *retriever* and the *generator*")
    print("separately, recovering the second paper that single-shot retrieval (\u00a74)")
    print("misses on this exact multi-hop question.")

### The evaluation harness (design)

`run_eval.py` runs both systems over every question and grades the answers with an
LLM-as-judge on **groundedness** (is every claim supported by the sources?) and
**correctness** (does it answer the question?), split by hop type, alongside the
agent's retrieval count. Results are cached and committed so they reproduce without
a key. The hypothesis the harness tests is precise: **the agent wins on multi-hop
questions — where §4 shows the retrieval gap lives — and roughly ties on single-hop
ones, at the cost of extra retrievals.**

In [ ]:
# If a committed answer-eval result exists, load and show it; otherwise state the plan.
ans_path = config.RESULTS_DIR / "answer_quality.json"
if ans_path.exists():
    ans = pd.DataFrame(json.load(open(ans_path)))
    display(ans.groupby(["hop", "system"])[["groundedness", "correctness", "n_retrievals"]].mean().round(2))
else:
    print("answer_quality.json not generated in this run (needs an API key once).")
    print("Run:  python run_eval.py   with ANTHROPIC_API_KEY set to produce it.")

## 7 · Limitations

- **The generated queries are built from their target documents' own words**, which
  structurally favours BM25. Mitigated, not eliminated, by measuring the *slope*
  across three query styles, by corroborating against the hand-written set (which
  does not share the bias), and by the whole result replicating on a corpus with
  zero paper overlap. See [`eval/ADJUDICATION.md`](../eval/ADJUDICATION.md).
- **Known-item retrieval is the easy end of the problem** — one right answer, no
  ambiguity, no near-duplicates. Absolute scores flatter every arm; the *ranking
  between* arms is what the set is for.
- **The curated set is still 11 questions.** Its bootstrap intervals overlap
  completely (BM25: [0.611, 0.962]), so it can corroborate an ordering but not
  establish one — and it is too small for the paired test to have useful power,
  which is why significance is reported on the generated set only.
- **The drift trigger has been evaluated at two points in time.** Excess-over-floor
  is the right *shape* of test; two readings are not a trend. And the floor itself
  expires whenever the embedder, chunk size or corpus size changes.
- **The chunk-size ablation covers two arms, not four.** The cross-encoder was
  excluded on cost (~2.6 s a query × 180 questions × 5 settings), so nothing here
  says how reranking interacts with chunk size.
- **One corpus, one embedding model, one CPU.** Nothing here says BM25 beats dense
  retrieval in general — it says it does on 2,012 arXiv abstracts with
  `all-MiniLM-L6-v2`. A stronger embedder, or full text, could reverse it.
- **Latency is single-threaded wall-clock, batch size one.** Fair between arms on
  identical hardware; not a throughput figure to plan capacity from. The reranker
  in particular would improve a lot with GPU batching.
- **Corpus is abstracts** — licence-safe. In practice `redistributable_count` came
  back **0**, so the full-text branch has never selected a paper.
- **LLM-as-judge** — a fair *relative* comparator (same judge, same rubric), not an
  absolute score.
- **Not hardened against prompt injection** — retrieved text is data, not
  instructions, in any real system.


## 8 · Conclusion — what I'd do about it

1. **Switch the first-stage retriever to BM25.** On this corpus the dense arm pays
   8× the latency for 0.227 less nDCG, and that gap is significant under a paired
   test with 52 wins to 1 loss. Not a tuning question — a default that was never
   tested, because nothing in the original project compared it to anything.
2. **Do not buy the cross-encoder.** 2.5 seconds a query to buy a difference that
   does not survive its own significance test.
3. **If you keep it anyway, cut its shortlist from 50 to 20.** Recall saturates at
   20, so candidates 21–50 are pure cost.
4. **Keep a dense arm as a fallback, not a replacement.** `s5` is unanswerable by
   any lexical method — the query term appears zero times in the corpus. Rare here,
   catastrophic when it happens. And *not* via naive RRF fusion, which loses the
   case too.
5. **Do not re-index yet.** The probe sits at its day-zero floor. The previous
   version of this list said "re-index now", on the strength of a metric that could
   not have said otherwise.
6. **The remaining multi-hop gap is still the agent's case.** `m4` needs two papers
   and no single retrieval strategy returns both.

The value of measuring is that the next step isn't a guess — including when the
number contradicts what the project was built to believe, and including when the
number itself turns out to be the thing that was wrong.


## 9 · Reproduce

```bash
python -m venv .venv && .venv/Scripts/activate
pip install -r requirements.txt
python run_pipeline.py          # build the index (local, no key)
pytest -q                       # 98 passed, 1 skipped — keyless
python run_retrieval_bench.py   # four arms + significance + k-sweep + drift (local, no key)
python run_ablation.py          # the chunk-size ablation (local, no key)
python run_eval.py              # retrieval metrics (local) + answer eval (needs key)
```

Everything in section 5 reproduces offline with no API key. Set
`ANTHROPIC_API_KEY` only to regenerate the naive-vs-agent answer comparison in
section 6 — after which it caches and reproduces without one.

The benchmark's runtime is dominated by the cross-encoder arm (~2.5 s a query,
scored twice — once for the arm comparison and once for the cutoff sweep).
Everything else is milliseconds.
